# How rising CO₂ changes the ionosphere's response to geomagnetic storms

**Roquetes (EB040), 1988 to 2026. Rebuilt analysis for the manuscript.**

This notebook follows the same order of work as the original project and the symposium slides:

1. Look at the data: time series, scatter plots, correlations
2. Build one pyGAM model for each ionospheric parameter (TEC, foF2, hmF2)
3. Assess each variable by removing it from the model
4. Explain every model with SHAP
5. Test a lagged Dst and a storm-days-only model
6. Add the Dst × CO₂ interaction and ask which variable is the modifier
7. **Put numbers on the modulation** (new)
8. Check the interaction against the raw data
9. Repeat with a CNN as an independent model
10. Check models: F10.7, a balanced period, daily mean Dst, no season
11. Future projection
12. All the numbers for the Results section in one place

**What is different from the original notebook, and why**

| Change | Why |
|---|---|
| Reads `merged_daily_filled.csv` | The rebuilt dataset (quality filter, hourly then daily means, CO₂ gaps filled, data to 2026) |
| Solar cycle 25 no longer ends in January 2023 | The old end date dropped every day after 2022 from the models |
| A season term (day of year) is in every model | The ionosphere and CO₂ both peak around May, so without it the season leaks into the CO₂ effect |
| The smoothness of the curves is chosen by leaving whole years out | The old interaction model was chosen by fit to its own training data and came out very wiggly |
| Models are scored on years they were not trained on | Neighbouring days are nearly identical, so a random split makes the test score too good |
| Uncertainty comes from a year-block bootstrap | Gives honest 95% intervals for every number reported |
| A linear-in-CO₂ interaction model is added | It turns the interaction into one number: the change in storm response per 10 ppm of CO₂ |

Run the cells from top to bottom. On Colab, upload `merged_daily_filled.csv` first.

## 0. Setup

In [ ]:
# Install the two libraries Colab does not have by default
import importlib, subprocess, sys
for pkg in ["pygam", "shap"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy import stats
from pygam import LinearGAM, s, te, l
import shap
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.feature_selection import mutual_info_regression

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

### Settings

Everything you may want to change is here. The storm levels depend on which Dst value is used:
with the **daily minimum**, a storm day is −50 nT or lower; with the **daily mean** the same storms look weaker, so the levels are smaller.

In [ ]:
DATA_FILE = "merged_daily_filled.csv"   # the dataset used in the paper (in the data folder of the repository)
for candidate in [DATA_FILE, os.path.join("data", DATA_FILE), os.path.join("ionosphere_data", "merged_daily.csv")]:
    if os.path.exists(candidate):     # also finds the file made by build_dataset.py
        DATA_FILE = candidate
        break

DST_COL = "dst_min"            # "dst_min" (daily minimum) or "dst_mean" (daily mean, as in the original)
START_YEAR = 1988              # first year with ionospheric data

PARAMS = {"TEC": "TECU", "foF2": "MHz", "hmF2": "km"}   # parameter: unit

# Reference conditions used when one variable is varied and the others are held fixed
CO2_REF = 390.0                # ppm, middle of the record
PHASE_REF = 0.35               # solar cycle phase, near solar maximum
DOY_REF = 80                   # day of year, March equinox
F107_REF = 120.0               # sfu, used only in the F10.7 check model
CO2_LEVELS = [350, 370, 390, 410, 430]        # ppm, inside the observed range
CO2_FUTURE = [450, 475, 500]                  # ppm, beyond the observed range

N_BOOT = 200                   # bootstrap repeats for the main numbers
N_BOOT_CHECK = 100             # bootstrap repeats for the check models
SHAP_N = 800                   # random days explained with SHAP
SHAP_STORM_N = 300             # extra storm days explained with SHAP
SHAP_BG = 100                  # background days for SHAP
RUN_CNN = True                 # set False to skip the CNN section
SEED = 42

FIG_DIR, TAB_DIR = "figures", "tables"
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)

if DST_COL == "dst_min":
    DST_QUIET, DST_LEVELS = -10.0, [-50.0, -100.0, -150.0]
    STORM_AT, QUIET_ABOVE = -50.0, -30.0
    DST_LABEL = "Daily minimum Dst (nT)"
else:
    DST_QUIET, DST_LEVELS = 0.0, [-30.0, -50.0, -80.0]
    STORM_AT, QUIET_ABOVE = -30.0, -15.0
    DST_LABEL = "Daily mean Dst (nT)"
MAIN_LEVEL = DST_LEVELS[1]     # the storm level quoted in the headline numbers

# Colours (colour-blind safe) and a common look for every figure
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"]
LS = ["-", "--", "-.", ":", (0, (5, 1))]
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False})

def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name + ".png"), bbox_inches="tight")
    plt.savefig(os.path.join(FIG_DIR, name + ".pdf"), bbox_inches="tight")

def savetab(table, name):
    table.to_csv(os.path.join(TAB_DIR, name + ".csv"))
    return table

## 1. Load the data

In [ ]:
df = pd.read_csv(DATA_FILE, parse_dates=["date"]).set_index("date").sort_index()

# Columns the models need
df["doy"] = df.index.dayofyear
df["co2c"] = (df["co2_ppm"] - CO2_REF) / 10.0                      # CO2 in units of 10 ppm, centred
df["f107_81"] = df["f107"].rolling(81, center=True, min_periods=40).mean()   # 81-day mean F10.7
df["dst_lag1"] = df[DST_COL].shift(1)                              # yesterday's Dst
df["year"] = df.index.year

full = df.copy()                       # 1974 onward, for the driver time series
df = df[df["year"] >= START_YEAR]      # 1988 onward, for everything else

print("Rows from", START_YEAR, ":", len(df), "|", df.index.min().date(), "to", df.index.max().date())
df[["co2_ppm", "dst_mean", "dst_min", "f107", "solar_cycle_phase", "TEC", "foF2", "hmF2"]].describe().round(2)

In [ ]:
# How many usable days does each year contribute?
per_year = df.groupby("year")[list(PARAMS)].count()
ax = per_year.plot(kind="bar", figsize=(12, 3.2), color=COL[:3], width=0.8)
ax.set_ylabel("Days with data"); ax.set_xlabel(""); ax.set_title("Days of ionospheric data per year")
savefig("fig_days_per_year"); plt.show()

storm_days = int((df[DST_COL] <= STORM_AT).sum())
print(f"Storm days ({DST_COL} <= {STORM_AT:.0f} nT): {storm_days} of {df[DST_COL].notna().sum()}")

## 2. Time series

The same first step as the original: look at how each variable behaves over time.
Vertical lines mark the start of solar cycles 22 to 25.

In [ ]:
CYCLE_STARTS = {22: "1986-09-01", 23: "1996-08-01", 24: "2008-12-01", 25: "2019-12-01"}
series = [("co2_ppm", "CO₂ (ppm)", full), (DST_COL, DST_LABEL, full), ("f107", "F10.7 (sfu)", full),
          ("TEC", "TEC (TECU)", df), ("foF2", "foF2 (MHz)", df), ("hmF2", "hmF2 (km)", df)]
fig, axes = plt.subplots(len(series), 1, figsize=(12, 13), sharex=True)
for ax, (col, label, data) in zip(axes, series):
    ax.plot(data.index, data[col], lw=0.4, color=COL[0])
    if col not in ("co2_ppm",):
        ax.plot(data.index, data[col].rolling(365, center=True, min_periods=120).mean(), lw=1.6, color=COL[1],
                label="1-year running mean")
    for start in CYCLE_STARTS.values():
        ax.axvline(pd.Timestamp(start), color="0.4", lw=0.8, ls="--")
    ax.set_ylabel(label)
axes[1].legend(loc="lower left")
axes[0].set_title("Drivers (from 1974) and ionospheric parameters (from 1988)")
savefig("fig01_time_series"); plt.show()

In [ ]:
# Each solar cycle laid over the others, against solar cycle phase.
# This shows that the cycles are not equally strong, which solar cycle phase alone cannot describe.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, p in zip(axes, PARAMS):
    for i, (cyc, g) in enumerate(df.groupby("solar_cycle")):
        sm = g[p].rolling(81, center=True, min_periods=30).mean()      # days without data stay empty, so gaps show as breaks
        ax.plot(g["solar_cycle_phase"], sm, color=COL[i % len(COL)], ls=LS[i % len(LS)], lw=1.5, label=f"Cycle {int(cyc)}")
    ax.set_xlabel("Solar cycle phase"); ax.set_ylabel(f"{p} ({PARAMS[p]}), 81-day mean")
axes[0].legend(); fig.suptitle("The same phase does not mean the same level in every cycle")
savefig("fig02_cycles_overlaid"); plt.show()

In [ ]:
# Stationarity (Augmented Dickey-Fuller test), as in the original analysis
try:
    from statsmodels.tsa.stattools import adfuller
    rows = []
    for col in ["co2_ppm", DST_COL, "TEC", "foF2", "hmF2"]:
        x = df[col].dropna()
        stat, pval = adfuller(x.values[::3], autolag="AIC")[:2]      # every third day keeps it quick
        rows.append({"variable": col, "ADF statistic": round(stat, 2), "p-value": round(pval, 4),
                     "stationary at 5%": "yes" if pval < 0.05 else "no"})
    display(savetab(pd.DataFrame(rows).set_index("variable"), "table_adf"))
except ImportError:
    print("statsmodels is not installed, skipping the ADF test")

## 3. Scatter plots and correlations

Scatter plots of each driver against each ionospheric parameter, then the correlation coefficients
(Pearson, Spearman and Kendall, as in the original).

In [ ]:
drivers = [(DST_COL, DST_LABEL), ("co2_ppm", "CO₂ (ppm)"), ("solar_cycle_phase", "Solar cycle phase")]
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for i, p in enumerate(PARAMS):
    for j, (dcol, dlab) in enumerate(drivers):
        d = df[[dcol, p]].dropna()
        axes[i, j].scatter(d[dcol], d[p], s=2, alpha=0.25, color=COL[0], rasterized=True)
        axes[i, j].set_xlabel(dlab); axes[i, j].set_ylabel(f"{p} ({PARAMS[p]})")
fig.suptitle("Each driver against each ionospheric parameter"); plt.tight_layout()
savefig("fig03_scatter"); plt.show()

In [ ]:
rows = []
for dcol, _ in drivers + [("f107", "")]:
    for p in PARAMS:
        d = df[[dcol, p]].dropna()
        rows.append({"driver": dcol, "parameter": p,
                     "Pearson": stats.pearsonr(d[dcol], d[p])[0],
                     "Spearman": stats.spearmanr(d[dcol], d[p])[0],
                     "Kendall": stats.kendalltau(d[dcol], d[p])[0]})
corr_table = pd.DataFrame(rows).set_index(["driver", "parameter"]).round(3)
display(savetab(corr_table, "table_correlations"))

d = df[[DST_COL, "co2_ppm", "solar_cycle_phase"]].dropna()
print("Correlation between the drivers themselves (Pearson):")
display(d.corr().round(3))

## 3b. Time series in three levels: whole record, per solar cycle, per year

As in the original analysis, the time series are examined at three levels:

1. **Whole record** (Section 2 above).
2. **Per solar cycle**: cycles 22, 23, 24 and 25, to see how each variable changes through a cycle and to compare the cycles.
   Cycle 21 is left out, as in the original, because the ionospheric data start in 1988.
3. **Per year**: each year on its own, to see the shorter patterns (the season, single storms) that are hidden in the long record.

The Augmented Dickey-Fuller test is run at each level.

In [ ]:
VARS = [(DST_COL, DST_LABEL), ("co2_ppm", "CO₂ (ppm)"), ("TEC", "TEC (TECU)"), ("foF2", "foF2 (MHz)"), ("hmF2", "hmF2 (km)")]
CYCLES = [22, 23, 24, 25]

def adf(x):
    # ADF statistic and p-value, or NaN if there are too few days
    x = pd.Series(x).dropna()
    if len(x) < 60 or x.nunique() < 5:
        return np.nan, np.nan
    stat, pval = adfuller(x.values, autolag="AIC")[:2]
    return stat, pval

# Per solar cycle: one figure per cycle, all five variables
rows = []
for cyc in CYCLES:
    g = full[full["solar_cycle"] == cyc]
    fig, axes = plt.subplots(len(VARS), 1, figsize=(12, 11), sharex=True)
    for ax, (col, label) in zip(axes, VARS):
        ax.plot(g.index, g[col], lw=0.5, color=COL[0])
        if col != "co2_ppm":
            ax.plot(g.index, g[col].rolling(81, center=True, min_periods=30).mean(), lw=1.6, color=COL[1], label="81-day mean")
        ax.set_ylabel(label)
        stat, pval = adf(g[col])
        rows.append({"solar cycle": cyc, "variable": col, "days": int(g[col].notna().sum()),
                     "ADF statistic": stat, "p-value": pval,
                     "stationary at 5%": ("yes" if pval < 0.05 else "no") if pd.notna(pval) else "too few days"})
    axes[1].legend(loc="upper left")
    axes[0].set_title(f"Solar cycle {cyc}: {g.index.min():%b %Y} to {g.index.max():%b %Y}")
    savefig(f"fig_timeseries_cycle{cyc}"); plt.show()
adf_cycle = pd.DataFrame(rows).set_index(["solar cycle", "variable"])
display(savetab(adf_cycle.round(4), "table_adf_per_cycle"))

In [ ]:
# Per year: for each variable, one panel per year
years_all = list(range(START_YEAR, df.index.year.max() + 1))
ncol = 6; nrow = int(np.ceil(len(years_all) / ncol))
rows = []
for col, label in VARS:
    fig, axes = plt.subplots(nrow, ncol, figsize=(18, 2.1 * nrow), sharey=True)
    for ax, yr in zip(axes.ravel(), years_all):
        g = df[df.index.year == yr][col]
        ax.plot(g.index.dayofyear, g.values, lw=0.6, color=COL[0])
        ax.set_xlim(1, 366)
        ax.set_title(str(yr) if g.notna().any() else f"{yr}: no data", fontsize=9); ax.tick_params(labelsize=7)
        stat, pval = adf(g)
        rows.append({"variable": col, "year": yr, "days": int(g.notna().sum()), "ADF statistic": stat, "p-value": pval,
                     "stationary at 5%": ("yes" if pval < 0.05 else "no") if pd.notna(pval) else "too few days"})
    for ax in axes.ravel()[len(years_all):]:
        ax.axis("off")
    fig.suptitle(f"{label}: time series of each year (x axis: day of year)", y=1.0)
    plt.tight_layout(); savefig(f"fig_timeseries_yearly_{col}"); plt.show()
adf_year = pd.DataFrame(rows).set_index(["variable", "year"])
savetab(adf_year.round(4), "table_adf_per_year")
summary = adf_year.groupby(level="variable")["stationary at 5%"].value_counts().unstack(fill_value=0)
print("How many years are stationary, by variable:"); display(summary)

## 3c. Scatter plots and correlations in three levels: whole record, per solar cycle, per year

The same three levels are used for the scatter plots and the Pearson, Spearman and Kendall coefficients, for every pair
in the original analysis:

- Dst against TEC, foF2 and hmF2
- CO₂ against TEC, foF2 and hmF2
- Dst against CO₂ (the two drivers; a strong relationship here would make the model unable to separate them)
- TEC, foF2 and hmF2 against each other

In [ ]:
PAIRS = [(DST_COL, "TEC"), (DST_COL, "foF2"), (DST_COL, "hmF2"),
         ("co2_ppm", "TEC"), ("co2_ppm", "foF2"), ("co2_ppm", "hmF2"),
         (DST_COL, "co2_ppm"),
         ("TEC", "foF2"), ("TEC", "hmF2"), ("foF2", "hmF2")]
LABEL = dict(VARS)

def correlations(d, x, y):
    d = d[[x, y]].dropna()
    if len(d) < 10:
        return {"days": len(d), "Pearson": np.nan, "Spearman": np.nan, "Kendall": np.nan}
    return {"days": len(d), "Pearson": stats.pearsonr(d[x], d[y])[0],
            "Spearman": stats.spearmanr(d[x], d[y])[0], "Kendall": stats.kendalltau(d[x], d[y])[0]}

# Level 1: whole record, all ten pairs
fig, axes = plt.subplots(2, 5, figsize=(20, 7.5))
rows = []
for ax, (x, y) in zip(axes.ravel(), PAIRS):
    d = df[[x, y]].dropna()
    ax.scatter(d[x], d[y], s=2, alpha=0.25, color=COL[0], rasterized=True)
    ax.set_xlabel(LABEL[x]); ax.set_ylabel(LABEL[y])
    k = PAIRS.index((x, y))
    ax.set_title(f"({'abcdefghij'[k]}) r = {correlations(df, x, y)['Pearson']:.2f}", loc="left", fontsize=10)
    rows.append({"pair": f"{x} vs {y}", **correlations(df, x, y)})
fig.suptitle("Whole record: every pair"); plt.tight_layout()
savefig("fig_scatter_whole_all_pairs"); plt.show()
corr_whole = pd.DataFrame(rows).set_index("pair")
display(savetab(corr_whole.round(3), "table_correlations_whole"))

In [ ]:
# Level 2: per solar cycle. One figure per pair, one panel per cycle
rows = []
for x, y in PAIRS:
    fig, axes = plt.subplots(1, len(CYCLES), figsize=(17, 3.6), sharex=True, sharey=True)
    for ax, cyc in zip(axes, CYCLES):
        g = df[df["solar_cycle"] == cyc]
        d = g[[x, y]].dropna()
        ax.scatter(d[x], d[y], s=3, alpha=0.3, color=COL[0], rasterized=True)
        c = correlations(g, x, y)
        ax.set_title(f"Cycle {cyc}: r = {c['Pearson']:.2f}", fontsize=10)
        ax.set_xlabel(LABEL[x])
        rows.append({"pair": f"{x} vs {y}", "solar cycle": cyc, **c})
    axes[0].set_ylabel(LABEL[y])
    fig.suptitle(f"{LABEL[y]} against {LABEL[x]}, per solar cycle"); plt.tight_layout()
    savefig(f"fig_scatter_cycle_{x}_{y}"); plt.show()
corr_cycle = pd.DataFrame(rows).set_index(["pair", "solar cycle"])
display(savetab(corr_cycle.round(3), "table_correlations_per_cycle"))

In [ ]:
# Level 3: per year. One figure per pair, one panel per year, and a figure of how the correlation changes from year to year
rows = []
for x, y in PAIRS:
    fig, axes = plt.subplots(nrow, ncol, figsize=(18, 2.2 * nrow))
    for ax, yr in zip(axes.ravel(), years_all):
        g = df[df.index.year == yr]
        d = g[[x, y]].dropna()
        ax.scatter(d[x], d[y], s=2, alpha=0.4, color=COL[0], rasterized=True)
        c = correlations(g, x, y)
        ax.set_title(f"{yr}: r = {c['Pearson']:.2f}" if pd.notna(c["Pearson"]) else f"{yr}: too few days", fontsize=8)
        ax.tick_params(labelsize=6)
        rows.append({"pair": f"{x} vs {y}", "year": yr, **c})
    for ax in axes.ravel()[len(years_all):]:
        ax.axis("off")
    fig.suptitle(f"{LABEL[y]} against {LABEL[x]}, per year", y=1.0)
    plt.tight_layout(); savefig(f"fig_scatter_yearly_{x}_{y}"); plt.show()
corr_year = pd.DataFrame(rows).set_index(["pair", "year"])
savetab(corr_year.round(3), "table_correlations_per_year")

# How the Pearson coefficient of each driver-parameter pair changes from year to year
fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
for ax, drv in zip(axes, [DST_COL, "co2_ppm"]):
    for i, p in enumerate(PARAMS):
        t = corr_year.loc[f"{drv} vs {p}"]
        ax.plot(t.index, t["Pearson"], marker="o", ms=3, color=COL[i], ls=LS[i], label=p)
    ax.axhline(0, color="0.4", lw=0.8)
    ax.set_xlabel("Year"); ax.set_title(f"{LABEL[drv]} against each parameter")
axes[0].set_ylabel("Pearson coefficient within the year"); axes[0].legend()
fig.suptitle("Correlation within each year"); savefig("fig_correlation_by_year"); plt.show()

### Storm activity per year, and correlations per solar cycle in one picture

In [ ]:
# Storm activity per year, and the correlation of each pair in each solar cycle
fig, ax = plt.subplots(figsize=(13, 4))
yrs = df[DST_COL].groupby(df.index.year)
n50 = yrs.apply(lambda s: (s <= STORM_AT).sum() * 365.25 / max(s.notna().sum(), 1))
n100 = yrs.apply(lambda s: (s <= -100).sum() * 365.25 / max(s.notna().sum(), 1))
cyc_of_year = df["solar_cycle"].groupby(df.index.year).agg(lambda s: s.mode().iloc[0] if s.notna().any() else np.nan)
for i, cyc in enumerate(CYCLES):
    sel = cyc_of_year == cyc
    ax.bar(n50.index[sel], n50[sel], color=COL[i], alpha=0.85, label=f"Cycle {cyc}: Dst ≤ {STORM_AT:.0f} nT")
ax.bar(n100.index, n100, color="0.2", width=0.45, label="Dst ≤ −100 nT (all cycles)")
ax.set_ylabel("Storm days per year"); ax.set_xlabel("Year"); ax.legend(fontsize=8, ncol=3)
ax.set_title("Number of storm days in each year (scaled to a full year where days are missing)")
savefig("fig_storm_days_per_year"); plt.show()
storm_rate = pd.DataFrame({"storm days per year (<= -50 nT)": n50, "strong storm days per year (<= -100 nT)": n100, "solar cycle": cyc_of_year})
savetab(storm_rate.round(1), "table_storm_days_per_year")

ct = pd.read_csv(os.path.join(TAB_DIR, "table_correlations_per_cycle.csv")) if "corr_cycle" not in globals() else corr_cycle.reset_index()
piv = ct.pivot(index="pair", columns="solar cycle", values="Pearson").loc[[f"{x} vs {y}" for x, y in PAIRS]]
fig, ax = plt.subplots(figsize=(14, 4.2))
xpos = np.arange(len(piv)); w = 0.2
for i, cyc in enumerate(CYCLES):
    ax.bar(xpos + (i - 1.5) * w, piv[cyc], width=w, color=COL[i], label=f"Cycle {cyc}")
ax.axhline(0, color="0.3", lw=0.8)
ax.set_xticks(xpos); ax.set_xticklabels([p.replace("dst_min", "Dst").replace("dst_mean", "Dst").replace("co2_ppm", "CO₂") for p in piv.index], rotation=20, fontsize=9)
ax.set_ylabel("Pearson coefficient"); ax.legend(ncol=4, fontsize=8)
ax.set_title("Correlation of each pair within each solar cycle")
plt.tight_layout(); savefig("fig_correlation_by_cycle"); plt.show()

## 3d. Violin plots

As in the original: violin plots show how the daily values are spread, not just their average.

1. **Each year**: one violin per year for Dst, CO₂, TEC, foF2 and hmF2, to see how the spread changes from year to year.
2. **Quiet days against storm days**: the days are split into quiet days (Dst above the quiet limit) and storm days
   (Dst at or below the storm limit set at the top). The days in between are left out so the two groups are clearly different.
   This compares the ionosphere when it is driven mainly by the Sun, the season and CO₂ with the ionosphere when a storm is added on top.

A Mann-Whitney test checks whether the storm-day and quiet-day values differ. Keep in mind that storms are more
common near solar maximum, so part of any difference here is the solar cycle; the models in the next sections separate the two.

In [ ]:
def violin(ax, groups, labels, colors):
    # Violin of each group, with the median and the middle 50% marked
    parts = ax.violinplot(groups, showextrema=False, widths=0.85)
    for body, c in zip(parts["bodies"], colors):
        body.set_facecolor(c); body.set_edgecolor("white"); body.set_alpha(0.75)
    for i, g in enumerate(groups, start=1):
        q1, med, q3 = np.percentile(g, [25, 50, 75])
        ax.vlines(i, q1, q3, color="0.15", lw=2.5)
        ax.scatter(i, med, color="white", edgecolor="0.15", s=18, zorder=3)
    ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels)

# 1. One violin per year, for all five variables (Dst and CO2 from 1974, as in the original)
for p in [DST_COL, "co2_ppm"] + list(PARAMS):
    d = (full if p in (DST_COL, "co2_ppm") else df)[[p, "year"]].dropna()
    yrs = sorted(d["year"].unique())
    fig, ax = plt.subplots(figsize=(14, 3.6))
    violin(ax, [d.loc[d["year"] == y, p].values for y in yrs], [str(y) for y in yrs], [COL[0]] * len(yrs))
    ax.tick_params(axis="x", rotation=90)
    ax.set_ylabel(LABEL[p]); ax.set_title(f"{LABEL[p]}: spread of daily values in each year (dot = median, bar = middle 50%)")
    savefig(f"fig_violin_yearly_{p}"); plt.show()

In [ ]:
# 2. Quiet days against storm days
cond = np.where(df[DST_COL] <= STORM_AT, "Storm", np.where(df[DST_COL] > QUIET_ABOVE, "Quiet", None))
rows = []
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, p in zip(axes, PARAMS):
    quiet = df.loc[(cond == "Quiet") & df[p].notna(), p].values
    storm = df.loc[(cond == "Storm") & df[p].notna(), p].values
    violin(ax, [quiet, storm], [f"Quiet\n(n = {len(quiet)})", f"Storm\n(n = {len(storm)})"], [COL[0], COL[1]])
    ax.set_ylabel(f"{p} ({PARAMS[p]})"); ax.set_title(p)
    pval = stats.mannwhitneyu(quiet, storm).pvalue
    rows.append({"parameter": p, "quiet days": len(quiet), "storm days": len(storm),
                 "median, quiet": np.median(quiet), "median, storm": np.median(storm),
                 "storm minus quiet (medians)": np.median(storm) - np.median(quiet),
                 "Mann-Whitney p-value": pval})
fig.suptitle(f"Quiet days (Dst > {QUIET_ABOVE:.0f} nT) against storm days (Dst ≤ {STORM_AT:.0f} nT)")
plt.tight_layout(); savefig("fig_violin_storm_vs_quiet"); plt.show()
violin_table = pd.DataFrame(rows).set_index("parameter")
display(savetab(violin_table.round(4), "table_storm_vs_quiet"))

## 4. The model

As in the original, one **pyGAM** model is built for each ionospheric parameter, with
solar cycle phase, Dst and CO₂ as predictors. Two things are added.

**Season.** A day-of-year term, joined at the ends so 31 December meets 1 January.
The ionosphere is highest around May and lowest in winter, and CO₂ also peaks in May every year.
Without this term the model gives part of the season to CO₂.

**A smooth CO₂ curve.** CO₂ rises almost steadily with time, so a very flexible CO₂ curve is really a
flexible curve of time, and it can absorb anything that changed over the years. CO₂ is therefore
limited to a few splines.

The model types used in this notebook:

| Name | Terms | Used for |
|---|---|---|
| `main` | s(phase) + s(Dst) + s(CO₂) + s(season) | Sections 5 to 7: the basic model |
| `te` | `main` + te(Dst, CO₂) | Section 8: is there an interaction, and what does it look like |
| `by` | s(phase) + s(Dst) + CO₂ (linear) + s(season) + s(Dst)·CO₂ | Section 9: the numbers, and the projection |

**How the smoothness is chosen.** The original used pyGAM's grid search, which picks the model with
the lowest GCV score. On this dataset GCV always picks the wiggliest model on offer, because
neighbouring days are so alike that a wiggly curve looks good to it. So the number of splines and the
smoothing penalty are chosen here by **year-blocked cross-validation**: the years are split into five
groups, the model is trained on four and tested on the fifth, and the setting with the lowest error on
the unseen years wins. The GCV score is still reported for every model.

In [ ]:
FEATURES = ["solar_cycle_phase", DST_COL, "co2_ppm", "doy"]       # what SHAP and the plots see
NAMES = ["Solar cycle phase", "Dst", "CO2", "Season"]
CO2_SPLINES = 5
LAMS = np.logspace(-2, 2, 5)          # smoothing penalties tried
K_GRID = (0.5, 0.75, 1.0, 1.5)        # multiplies the spline counts
N_FOLDS = 5

def design(data, dst_col=None, solar=None):
    # Model input: phase, Dst, CO2, day of year, CO2 in 10-ppm units, and optionally a solar index
    return ["solar_cycle_phase", dst_col or DST_COL, "co2_ppm", "doy", "co2c"] + ([solar] if solar else [])

def get_xy(data, p, dst_col=None, solar=None):
    cols = design(data, dst_col, solar)
    d = data[cols + [p]].dropna()
    return d, d[cols].values.astype(float), d[p].values.astype(float), d.index.year.values

def build_terms(kind, K=1.0, solar=False, drop=None):
    # Returns the pyGAM terms and a label for each one
    parts, labels = [], []
    if drop != "phase":
        parts.append(s(0, n_splines=int(round(12 * K)))); labels.append("phase")
    if drop != "dst":
        parts.append(s(1, n_splines=int(round(8 * K)))); labels.append("dst")
    if drop != "co2":
        parts.append(s(2, n_splines=CO2_SPLINES) if kind in ("main", "te") else l(2)); labels.append("co2")
    if drop != "season":
        parts.append(s(3, n_splines=int(round(10 * K)), basis="cp", edge_knots=[0.5, 366.5])); labels.append("season")
    if kind == "te":
        parts.append(te(1, 2, n_splines=[6, 4])); labels.append("dst x co2")
    if kind == "by":
        parts.append(s(1, by=4, n_splines=6)); labels.append("dst x co2")
    if solar:
        parts.append(s(5, n_splines=8)); labels.append("solar")
    terms = parts[0]
    for part in parts[1:]:
        terms = terms + part
    return terms, labels

def fit_gam(X, y, kind, settings, solar=False, drop=None):
    # settings = (spline multiplier, smoothing penalty), chosen once per parameter by choose_settings
    K, lam = settings
    terms, labels = build_terms(kind, K, solar, drop)
    gam = LinearGAM(terms); gam.lam = lam          # one smoothing penalty shared by all terms
    gam.fit(X, y)
    gam.labels = labels
    return gam

def year_folds(years):
    # Five groups of whole years
    return list(GroupKFold(n_splits=N_FOLDS).split(np.zeros(len(years)), groups=years))

def unseen_year_predictions(X, y, years, kind, settings, solar=False, drop=None):
    # Every day is predicted by a model that never saw that day's year
    pred = np.empty(len(y))
    for train, test in year_folds(years):
        pred[test] = fit_gam(X[train], y[train], kind, settings, solar, drop).predict(X[test])
    return pred

def choose_settings(X, y, years):
    # Try every combination and keep the one with the lowest error on unseen years
    rows = []
    for K in K_GRID:
        for lam in LAMS:
            pred = unseen_year_predictions(X, y, years, "main", (K, lam))
            rows.append({"spline multiplier": K, "lambda": lam, "RMSE on unseen years": np.sqrt(np.mean((y - pred) ** 2))})
    table = pd.DataFrame(rows).sort_values("RMSE on unseen years").reset_index(drop=True)
    return (float(table.loc[0, "spline multiplier"]), float(table.loc[0, "lambda"])), table

def gam_stats(gam):
    st = gam.statistics_
    return {"deviance explained": st["pseudo_r2"]["explained_deviance"], "AIC": st["AIC"],
            "GCV": st["GCV"], "effective dof": st["edof"]}

def ref_rows(dst, co2, phase=PHASE_REF, doy=DOY_REF, solar_value=None):
    # Rows of model input in which only Dst and CO2 vary
    dst, co2 = np.broadcast_arrays(np.atleast_1d(dst).astype(float), np.atleast_1d(co2).astype(float))
    cols = [np.full(dst.shape, phase), dst, co2, np.full(dst.shape, float(doy)), (co2 - CO2_REF) / 10.0]
    if solar_value is not None:
        cols.append(np.full(dst.shape, float(solar_value)))
    return np.column_stack(cols)

def split_years(years, test_size=0.3):
    # One split with whole years in the test set (used only for the CNN, which needs a validation set)
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    train, test = next(gss.split(np.zeros(len(years)), groups=years))
    return train, test

## 5. A pyGAM model for each ionospheric parameter

For each of TEC, foF2 and hmF2:

- the smoothness is chosen by year-blocked cross-validation,
- the model is fitted on all the data,
- it is scored on **unseen years**: every day is predicted by a model that was not shown that day's year,
- the predicted values are compared with the actual values,
- the partial dependence of each variable is plotted with its 95% confidence interval.

In [ ]:
models, data_store, SET, fit_rows = {}, {}, {}, []
for p in PARAMS:
    d, X, y, years = get_xy(df, p)
    SET[p], cv_table = choose_settings(X, y, years)
    gam = fit_gam(X, y, "main", SET[p])
    unseen = unseen_year_predictions(X, y, years, "main", SET[p])
    fit_rows.append({"parameter": p, "days": len(y), **gam_stats(gam),
                     "R2 same data": r2_score(y, gam.predict(X)),
                     "R2 unseen years": r2_score(y, unseen),
                     "RMSE unseen years": np.sqrt(mean_squared_error(y, unseen)),
                     "spline multiplier": SET[p][0], "lambda": SET[p][1]})
    models[p] = gam
    data_store[p] = dict(d=d, X=X, y=y, years=years, unseen=unseen)
    print(f"{p}: best three settings"); display(cv_table.head(3).round(4))

fit_table = pd.DataFrame(fit_rows).set_index("parameter")
display(savetab(fit_table.round(4), "table_main_models"))

In [ ]:
# Predicted against actual, for years the model did not see
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, p in zip(axes, PARAMS):
    st = data_store[p]
    actual, pred = st["y"], st["unseen"]
    ax.scatter(actual, pred, s=3, alpha=0.3, color=COL[0], rasterized=True)
    lim = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
    ax.plot(lim, lim, color="0.3", lw=1)
    ax.set_xlabel(f"Actual {p} ({PARAMS[p]})"); ax.set_ylabel(f"Predicted {p} ({PARAMS[p]})")
    ax.set_title(f"{p}: R² = {r2_score(actual, pred):.2f} on unseen years")
savefig("fig04_predicted_vs_actual"); plt.show()

In [ ]:
def plot_partial_dependence(gam, p, tag):
    # One panel per variable, with the 95% confidence interval
    xlabels = {"phase": "Solar cycle phase", "dst": DST_LABEL, "co2": "CO₂ (ppm)", "season": "Day of year",
               "solar": "F10.7, 81-day mean (sfu)"}
    shown = [(i, lab) for i, lab in enumerate(gam.labels) if lab in xlabels]
    fig, axes = plt.subplots(1, len(shown), figsize=(4.2 * len(shown), 3.6))
    for ax, (i, lab) in zip(np.atleast_1d(axes), shown):
        XX = gam.generate_X_grid(term=i)
        feat = gam.terms[i].feature
        pdep, conf = gam.partial_dependence(term=i, X=XX, width=0.95)
        ax.plot(XX[:, feat], pdep, color=COL[0], lw=2)
        ax.fill_between(XX[:, feat], conf[:, 0], conf[:, 1], color=COL[0], alpha=0.2)
        ax.axhline(0, color="0.4", lw=0.8)
        ax.set_xlabel(xlabels[lab]); ax.set_ylabel(f"Effect on {p} ({PARAMS[p]})")
    fig.suptitle(f"{p}: effect of each variable, with 95% confidence interval"); plt.tight_layout()
    savefig(f"fig05_partial_dependence_{p}_{tag}"); plt.show()

for p in PARAMS:
    plot_partial_dependence(models[p], p, "main")

## 6. Variable assessment

As in the original: remove one variable at a time, refit, and see how much worse the model gets.
The table also shows the loss in R² on unseen years, which is the fairest measure.

In [ ]:
rows = []
for p in PARAMS:
    st, gam = data_store[p], models[p]
    full_stats = gam_stats(gam)
    full_r2 = r2_score(st["y"], st["unseen"])
    for drop in ["phase", "dst", "co2", "season"]:
        g = fit_gam(st["X"], st["y"], "main", SET[p], drop=drop)
        unseen = unseen_year_predictions(st["X"], st["y"], st["years"], "main", SET[p], drop=drop)
        s2 = gam_stats(g)
        rows.append({"parameter": p, "removed": drop,
                     "deviance explained": s2["deviance explained"],
                     "loss in deviance explained (%)": 100 * (full_stats["deviance explained"] - s2["deviance explained"]),
                     "AIC increase": s2["AIC"] - full_stats["AIC"],
                     "GCV increase": s2["GCV"] - full_stats["GCV"],
                     "loss in R2 on unseen years": full_r2 - r2_score(st["y"], unseen)})
assess = pd.DataFrame(rows).set_index(["parameter", "removed"])
display(savetab(assess.round(4), "table_variable_assessment"))

## 7. SHAP on the models

SHAP is applied to **every model in this notebook** with the same function, so the results can be compared.

For each model it gives:

- the summary plot,
- the mean absolute SHAP value of each variable, and its share of the total (the relative contribution),
- dependence plots coloured by a second variable, to see interactions,
- the mutual information scores used in the original (how much one variable tells us about another variable's SHAP values),
- **a number for the modulation**: on storm days, how much the Dst SHAP value changes for every 10 ppm of CO₂;
- **the same number for solar activity**: on high-solar days (solar cycle phase 0.25 to 0.55, or the top third of F10.7),
  how much the solar SHAP value changes for every 10 ppm of CO₂. This checks the solar side of space weather in every model.

SHAP is run on a random sample of days plus extra storm days, because storm days are rare.
The importance figures use only the random sample, so they are not biased towards storms.

In [ ]:
def shap_sample(X4, dst_values):
    # A random sample of days, plus extra storm days
    rng = np.random.default_rng(SEED)
    idx_random = rng.choice(len(X4), size=min(SHAP_N, len(X4)), replace=False)
    storm_idx = np.where(dst_values <= STORM_AT)[0]
    storm_idx = np.setdiff1d(storm_idx, idx_random)
    idx_storm = rng.choice(storm_idx, size=min(SHAP_STORM_N, len(storm_idx)), replace=False)
    background = X4[rng.choice(len(X4), size=min(SHAP_BG, len(X4)), replace=False)]
    return idx_random, idx_storm, background

def run_shap(predict_fn, X4, names, title, tag, unit, dependence=True):
    # X4 holds the variables in the order of `names`; column 0 is the solar variable, column 1 Dst and column 2 CO2
    idx_random, idx_storm, background = shap_sample(X4, X4[:, 1])
    explainer = shap.KernelExplainer(predict_fn, background)
    Xr, Xs = X4[idx_random], X4[idx_storm]
    sv_r = np.array(explainer.shap_values(Xr, silent=True)).reshape(len(Xr), -1)
    sv_s = np.array(explainer.shap_values(Xs, silent=True)).reshape(len(Xs), -1)
    Xall, sv_all = np.vstack([Xr, Xs]), np.vstack([sv_r, sv_s])

    shap.summary_plot(sv_r, Xr, feature_names=names, show=False)
    plt.title(f"{title}: SHAP summary"); savefig(f"fig_shap_summary_{tag}"); plt.show()

    mean_abs = np.abs(sv_r).mean(axis=0)
    importance = pd.DataFrame({"mean |SHAP|": mean_abs, "share (%)": 100 * mean_abs / mean_abs.sum()}, index=names)

    if dependence:
        pairs = [("Dst", "CO2"), ("CO2", "Dst"), (names[0], "CO2")]
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        for ax, (a, b) in zip(axes, pairs):
            shap.dependence_plot(a, sv_all, Xall, feature_names=names, interaction_index=b, ax=ax, show=False)
            ax.set_ylabel(f"SHAP value of {a} ({unit})")
        fig.suptitle(f"{title}: SHAP dependence, coloured by a second variable")
        savefig(f"fig_shap_dependence_{tag}"); plt.show()

    # Mutual information, as in the original: does one variable tell us about another's SHAP values?
    i_phase, i_dst, i_co2 = 0, names.index("Dst"), names.index("CO2")
    mi = lambda feature, shap_col: mutual_info_regression(Xall[:, [feature]], sv_all[:, shap_col], random_state=SEED)[0]
    mi_scores = {"CO2 -> Dst's impact": mi(i_co2, i_dst), "Dst -> CO2's impact": mi(i_dst, i_co2),
                 "CO2 -> solar's impact": mi(i_co2, i_phase), "solar -> CO2's impact": mi(i_phase, i_co2)}

    # The modulation as a number: on storm days, slope of the Dst SHAP value against CO2
    storm = Xall[:, i_dst] <= STORM_AT
    slope, lo, hi = np.nan, np.nan, np.nan
    if storm.sum() >= 30:
        x, yv = Xall[storm, i_co2], sv_all[storm, i_dst]
        slope = np.polyfit(x, yv, 1)[0] * 10
        rng = np.random.default_rng(SEED)
        boots = [np.polyfit(x[j], yv[j], 1)[0] * 10 for j in (rng.integers(0, len(x), len(x)) for _ in range(500))]
        lo, hi = np.percentile(boots, [2.5, 97.5])
    # The same for solar activity: on high-solar days, slope of the solar SHAP value against CO2
    sol = Xall[:, i_phase]
    high = (sol >= 0.25) & (sol <= 0.55) if names[0] == "Solar cycle phase" else sol >= np.percentile(sol, 67)
    x, yv = Xall[high, i_co2], sv_all[high, i_phase]
    s_slope = np.polyfit(x, yv, 1)[0] * 10
    rng = np.random.default_rng(SEED)
    sb = [np.polyfit(x[j], yv[j], 1)[0] * 10 for j in (rng.integers(0, len(x), len(x)) for _ in range(500))]
    s_lo, s_hi = np.percentile(sb, [2.5, 97.5])
    return {"importance": importance, "mi": mi_scores, "storm_days_explained": int(storm.sum()),
            "dst_shap_slope_per_10ppm": (slope, lo, hi),
            "high_solar_days_explained": int(high.sum()), "solar_shap_slope_per_10ppm": (s_slope, s_lo, s_hi)}

def gam_predictor(gam, solar=False):
    # SHAP sees phase, Dst, CO2, day of year (and the solar index); the 10-ppm CO2 column is rebuilt here
    def predict(A):
        A = np.asarray(A, dtype=float)
        cols = [A[:, 0], A[:, 1], A[:, 2], A[:, 3], (A[:, 2] - CO2_REF) / 10.0]
        if solar:
            cols.append(A[:, 4])
        return gam.predict(np.column_stack(cols))
    return predict

def show_shap(result, title):
    print(title)
    display(result["importance"].round(3))
    print("Mutual information scores:", {k: round(v, 3) for k, v in result["mi"].items()})
    sl, lo, hi = result["dst_shap_slope_per_10ppm"]
    print(f"Storm days explained: {result['storm_days_explained']}. "
          f"Change in the Dst SHAP value per +10 ppm CO2: {sl:+.3f} [{lo:+.3f}, {hi:+.3f}]")
    sl, lo, hi = result["solar_shap_slope_per_10ppm"]
    print(f"High-solar days explained: {result['high_solar_days_explained']}. "
          f"Change in the solar SHAP value per +10 ppm CO2: {sl:+.3f} [{lo:+.3f}, {hi:+.3f}]\n")

In [ ]:
shap_results = {}
for p in PARAMS:
    st = data_store[p]
    res = run_shap(gam_predictor(models[p]), st["X"][:, :4], NAMES, f"{p}, main model", f"{p}_main", PARAMS[p])
    shap_results[(p, "main")] = res
    show_shap(res, f"--- {p}, main model ---")

In the main model there is no interaction term, so the Dst SHAP value cannot depend on CO₂.
The slope printed above should be close to zero here. It is the baseline against which the
interaction models in Sections 8 and 9 are compared.

## 8. Follow-up checks from the original: lagged Dst, and storm days only

**Lagged Dst.** Does the ionosphere respond to yesterday's Dst better than to today's?

**Storm days only.** Does Dst become more important when only storm days are modelled?

In [ ]:
rows = []
for p in PARAMS:
    cols = ["solar_cycle_phase", DST_COL, "co2_ppm", "doy", "co2c", "dst_lag1", p]
    d = df[cols].dropna()
    y = d[p].values
    X_same = d[["solar_cycle_phase", DST_COL, "co2_ppm", "doy", "co2c"]].values
    X_lag = d[["solar_cycle_phase", "dst_lag1", "co2_ppm", "doy", "co2c"]].values
    g_same, g_lag = fit_gam(X_same, y, "main", SET[p]), fit_gam(X_lag, y, "main", SET[p])
    rows.append({"parameter": p, "days": len(y),
                 "deviance explained, same-day Dst": gam_stats(g_same)["deviance explained"],
                 "deviance explained, lagged Dst": gam_stats(g_lag)["deviance explained"],
                 "GCV, same-day Dst": gam_stats(g_same)["GCV"], "GCV, lagged Dst": gam_stats(g_lag)["GCV"]})
lag_table = pd.DataFrame(rows).set_index("parameter")
lag_table["better"] = np.where(lag_table["GCV, lagged Dst"] < lag_table["GCV, same-day Dst"], "lagged", "same day")
display(savetab(lag_table.round(4), "table_lagged_dst"))

In [ ]:
rows = []
for p in PARAMS:
    d, X, y, years = get_xy(df[df[DST_COL] <= STORM_AT], p)
    g = fit_gam(X, y, "main", SET[p])          # same model type, storm days only
    base = gam_stats(g)
    for drop in ["phase", "dst", "co2", "season"]:
        g2 = fit_gam(X, y, "main", SET[p], drop=drop)
        rows.append({"parameter": p, "storm days": len(y), "deviance explained (full)": base["deviance explained"],
                     "removed": drop,
                     "loss in deviance explained (%)": 100 * (base["deviance explained"] - gam_stats(g2)["deviance explained"])})
storm_table = pd.DataFrame(rows).set_index(["parameter", "removed"])
display(savetab(storm_table.round(3), "table_storm_days_only"))

## 9. The main question: does CO₂ change how the ionosphere responds to geomagnetic storms?

As in the original, an interaction term between Dst and CO₂ is added with a tensor spline, `te(Dst, CO₂)`.

First: does the interaction improve the model at all? The table compares the model with and without it.
A lower GCV and AIC, and a higher R² on unseen years, mean the interaction is worth having.

In [ ]:
te_models, rows = {}, []
for p in PARAMS:
    st = data_store[p]
    g = fit_gam(st["X"], st["y"], "te", SET[p])
    unseen = unseen_year_predictions(st["X"], st["y"], st["years"], "te", SET[p])
    te_models[p] = g
    a, b = gam_stats(models[p]), gam_stats(g)
    rows.append({"parameter": p,
                 "deviance explained, no interaction": a["deviance explained"], "deviance explained, with interaction": b["deviance explained"],
                 "GCV change": b["GCV"] - a["GCV"], "AIC change": b["AIC"] - a["AIC"],
                 "R2 unseen years, no interaction": r2_score(st["y"], st["unseen"]),
                 "R2 unseen years, with interaction": r2_score(st["y"], unseen)})
te_table = pd.DataFrame(rows).set_index("parameter")
display(savetab(te_table.round(4), "table_interaction_gain"))

### The interaction surface

The three views from the original: a 3D surface, a heatmap and a contour plot of the interaction term.
The colour scale is centred on zero: one colour means the interaction raises the parameter, the other that it lowers it.

In [ ]:
dst_grid = np.linspace(np.nanpercentile(df[DST_COL], 0.2), np.nanpercentile(df[DST_COL], 99.8), 60)
co2_grid = np.linspace(df["co2_ppm"].min(), df["co2_ppm"].max(), 60)
DD, CC = np.meshgrid(dst_grid, co2_grid)

def interaction_surface(gam):
    i = gam.labels.index("dst x co2")
    XX = ref_rows(DD.ravel(), CC.ravel())
    return gam.partial_dependence(term=i, X=XX).reshape(DD.shape)

for p in PARAMS:
    Z = interaction_surface(te_models[p])
    vmax = np.abs(Z).max()
    fig = plt.figure(figsize=(16, 4.4))
    ax = fig.add_subplot(1, 3, 1, projection="3d")
    ax.plot_surface(DD, CC, Z, cmap="RdBu_r", vmin=-vmax, vmax=vmax, linewidth=0, antialiased=True)
    ax.set_xlabel(DST_LABEL); ax.set_ylabel("CO₂ (ppm)"); ax.set_zlabel(f"Effect on {p} ({PARAMS[p]})")
    ax2 = fig.add_subplot(1, 3, 2)
    im = ax2.pcolormesh(DD, CC, Z, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto", rasterized=True)
    ax2.set_xlabel(DST_LABEL); ax2.set_ylabel("CO₂ (ppm)"); ax2.grid(False)
    fig.colorbar(im, ax=ax2, label=f"Effect on {p} ({PARAMS[p]})")
    ax3 = fig.add_subplot(1, 3, 3)
    cs = ax3.contour(DD, CC, Z, levels=10, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax3.clabel(cs, inline=True, fontsize=7); ax3.set_xlabel(DST_LABEL); ax3.set_ylabel("CO₂ (ppm)")
    fig.suptitle(f"{p}: the Dst × CO₂ interaction term (surface, heatmap, contours)")
    savefig(f"fig06_interaction_surface_{p}"); plt.show()

### Which variable is the modifier?

The same two cases as the original.

**Case 1, CO₂ as the modifier:** predicted value against Dst, with CO₂ held at fixed levels.
If CO₂ did not change the storm response, the lines would be parallel.

**Case 2, Dst as the modifier:** predicted value against CO₂, with Dst held at fixed levels.

Solar cycle phase and season are held at the reference values set at the top.

In [ ]:
dst_line = np.linspace(-200 if DST_COL == "dst_min" else -120, 20, 80)
co2_line = np.linspace(df["co2_ppm"].min(), df["co2_ppm"].max(), 80)
for p in PARAMS:
    g = te_models[p]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for i, c in enumerate(CO2_LEVELS):
        axes[0].plot(dst_line, g.predict(ref_rows(dst_line, c)), color=COL[i % len(COL)], ls=LS[i % len(LS)], lw=1.8, label=f"CO₂ = {c} ppm")
    axes[0].set_xlabel(DST_LABEL); axes[0].set_ylabel(f"Predicted {p} ({PARAMS[p]})")
    axes[0].set_title("Case 1: CO₂ as the modifier"); axes[0].legend()
    for i, dlev in enumerate([DST_QUIET] + DST_LEVELS):
        axes[1].plot(co2_line, g.predict(ref_rows(dlev, co2_line)), color=COL[i % len(COL)], ls=LS[i % len(LS)], lw=1.8, label=f"Dst = {dlev:.0f} nT")
    axes[1].set_xlabel("CO₂ (ppm)"); axes[1].set_ylabel(f"Predicted {p} ({PARAMS[p]})")
    axes[1].set_title("Case 2: Dst as the modifier"); axes[1].legend()
    fig.suptitle(f"{p}: who modifies whom (phase = {PHASE_REF}, day of year = {DOY_REF})")
    savefig(f"fig07_modifier_{p}"); plt.show()

In [ ]:
# SHAP on the interaction models
for p in PARAMS:
    st = data_store[p]
    res = run_shap(gam_predictor(te_models[p]), st["X"][:, :4], NAMES, f"{p}, interaction model", f"{p}_te", PARAMS[p])
    shap_results[(p, "te")] = res
    show_shap(res, f"--- {p}, interaction model ---")

## 10. Putting numbers on the modulation

The plots above show the shape of the interaction. To state it as a number, two definitions are used.

**Storm response.** The difference between the predicted value on a storm day and on a quiet day,
with everything else the same:

> storm response at a CO₂ level = prediction(Dst = storm level) − prediction(Dst = quiet level)

**Modulation.** How much the storm response changes when CO₂ rises by 10 ppm.
A modulation of zero means CO₂ does not change the storm response.

To get one clear number, the interaction is made **linear in CO₂** (the `by` model): the Dst curve is
allowed to change in proportion to CO₂. The modulation is then the same at every CO₂ level, and it
has a 95% interval.

**Uncertainty.** The intervals come from a bootstrap that resamples **whole years**. Days within a
year are not independent, so resampling single days would make the intervals too narrow.

If the 95% interval of the modulation does not include zero, the data support a real change in the storm response.
If it includes zero, the data cannot tell the change apart from no change.

In [ ]:
DST_BOOT = np.unique(np.concatenate([np.linspace(-200 if DST_COL == "dst_min" else -120, 20, 45), DST_LEVELS, [DST_QUIET]]))
CO2_BOOT = np.arange(340, 521, 5).astype(float)
IQ = int(np.argmin(np.abs(DST_BOOT - DST_QUIET)))
I_REF, I_REF10 = int(np.where(CO2_BOOT == 390)[0][0]), int(np.where(CO2_BOOT == 400)[0][0])

def grid_predict(gam, solar_value=None):
    # Predictions on the whole Dst x CO2 grid, at the reference phase and season
    D, Cg = np.meshgrid(DST_BOOT, CO2_BOOT, indexing="ij")
    return gam.predict(ref_rows(D.ravel(), Cg.ravel(), solar_value=solar_value)).reshape(D.shape)

def bootstrap_by_model(X, y, years, settings, solar=False, drop=None, n_boot=N_BOOT, solar_value=None):
    # Fit the linear-in-CO2 interaction model, then refit it on resampled years
    gam = fit_gam(X, y, "by", settings, solar=solar, drop=drop)
    point = grid_predict(gam, solar_value)
    rng = np.random.default_rng(SEED)
    uniq = np.unique(years); where = {u: np.where(years == u)[0] for u in uniq}
    boots = []
    for _ in range(n_boot):
        pick = np.concatenate([where[u] for u in rng.choice(uniq, size=len(uniq), replace=True)])
        try:
            g = fit_gam(X[pick], y[pick], "by", settings, solar=solar, drop=drop)
            boots.append(grid_predict(g, solar_value))
        except Exception:
            continue
    return gam, point, np.array(boots)

def ci(values):
    return np.percentile(values, 2.5, axis=0), np.percentile(values, 97.5, axis=0)

def response(grid, level):
    # Storm response at every CO2 on the grid: storm level minus quiet level
    i = int(np.argmin(np.abs(DST_BOOT - level)))
    return grid[..., i, :] - grid[..., IQ, :]

def modulation(grid, level):
    # Change in the storm response per +10 ppm CO2
    r = response(grid, level)
    return r[..., I_REF10] - r[..., I_REF]

def co2_effect(grid):
    # Change in the quiet-day value per +10 ppm CO2
    return grid[..., IQ, I_REF10] - grid[..., IQ, I_REF]

def fmt(value, lo, hi, digits=3):
    return f"{value:+.{digits}f} [{lo:+.{digits}f}, {hi:+.{digits}f}]"

def number_table(point, boots, p):
    # The table of numbers for one model
    rows = []
    lo, hi = ci(co2_effect(boots))
    rows.append({"quantity": "Change in quiet-day value per +10 ppm CO2", "estimate [95% interval]": fmt(co2_effect(point), lo, hi),
                 "interval excludes zero": bool(lo > 0 or hi < 0)})
    for level in DST_LEVELS:
        r, rb = response(point, level), response(boots, level)
        for c in (350, 430):
            j = int(np.where(CO2_BOOT == c)[0][0]); lo, hi = ci(rb[:, j])
            rows.append({"quantity": f"Storm response at Dst = {level:.0f} nT, CO2 = {c} ppm",
                         "estimate [95% interval]": fmt(r[j], lo, hi), "interval excludes zero": bool(lo > 0 or hi < 0)})
        m, mb = modulation(point, level), modulation(boots, level); lo, hi = ci(mb)
        rows.append({"quantity": f"MODULATION at Dst = {level:.0f} nT: change in storm response per +10 ppm CO2",
                     "estimate [95% interval]": fmt(m, lo, hi), "interval excludes zero": bool(lo > 0 or hi < 0)})
    t = pd.DataFrame(rows).set_index("quantity"); t.columns.name = f"{p} ({PARAMS[p]})"
    return t

In [ ]:
by_models, by_point, by_boots, number_tables = {}, {}, {}, {}
t0 = time.time()
for p in PARAMS:
    st = data_store[p]
    by_models[p], by_point[p], by_boots[p] = bootstrap_by_model(st["X"], st["y"], st["years"], SET[p])
    number_tables[p] = savetab(number_table(by_point[p], by_boots[p], p), f"table_modulation_{p}")
    print(f"{p}: {len(by_boots[p])} bootstrap fits, deviance explained {gam_stats(by_models[p])['deviance explained']:.3f}")
    display(number_tables[p])
print(f"Time: {time.time() - t0:.0f} s")

In [ ]:
# Figure: the storm response at each CO2 level, and the modulation along the Dst axis
obs = (CO2_BOOT >= df["co2_ppm"].min()) & (CO2_BOOT <= df["co2_ppm"].max())
for p in PARAMS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for i, level in enumerate(DST_LEVELS):
        r, (lo, hi) = response(by_point[p], level), ci(response(by_boots[p], level))
        axes[0].plot(CO2_BOOT[obs], r[obs], color=COL[i], ls=LS[i], lw=2, label=f"Dst = {level:.0f} nT")
        axes[0].fill_between(CO2_BOOT[obs], lo[obs], hi[obs], color=COL[i], alpha=0.15)
    axes[0].axhline(0, color="0.4", lw=0.8)
    axes[0].set_xlabel("CO₂ (ppm)"); axes[0].set_ylabel(f"Storm response of {p} ({PARAMS[p]})")
    axes[0].set_title("Storm response at each CO₂ level"); axes[0].legend()

    full_grid = by_boots[p][:, :, I_REF10] - by_boots[p][:, :, I_REF]
    m_boot = full_grid - full_grid[:, [IQ]]
    m_point = (by_point[p][:, I_REF10] - by_point[p][:, I_REF]); m_point = m_point - m_point[IQ]
    lo, hi = ci(m_boot)
    axes[1].plot(DST_BOOT, m_point, color=COL[0], lw=2)
    axes[1].fill_between(DST_BOOT, lo, hi, color=COL[0], alpha=0.2)
    axes[1].axhline(0, color="0.4", lw=0.8)
    axes[1].set_xlabel(DST_LABEL); axes[1].set_ylabel(f"Change in Dst effect per +10 ppm CO₂ ({PARAMS[p]})")
    axes[1].set_title("Modulation along the Dst axis (relative to a quiet day)")
    fig.suptitle(f"{p}: storm response and its modulation by CO₂, with 95% intervals")
    savefig(f"fig08_modulation_{p}"); plt.show()

In [ ]:
# SHAP on the models that give the numbers
for p in PARAMS:
    st = data_store[p]
    res = run_shap(gam_predictor(by_models[p]), st["X"][:, :4], NAMES, f"{p}, linear-in-CO₂ interaction model", f"{p}_by", PARAMS[p])
    shap_results[(p, "by")] = res
    show_shap(res, f"--- {p}, linear-in-CO2 interaction model ---")

## 10b. The same calculation on other days: different solar phases and seasons

The numbers in Section 10 were worked out for one kind of day: solar cycle phase 0.35 and the March equinox.
One day is not enough to base the results on, so the calculation is repeated for **16 kinds of day**, with CO₂ at the same
levels (350 and 430 ppm) and the same storm (−100 nT against a quiet −10 nT):

| Solar phase | Why |
|---|---|
| Minimum (0.05) | The Sun is quiet; the ionosphere is at its lowest |
| Rising (0.20) | The Sun is getting stronger |
| Maximum (0.40) | The Sun is strongest; the ionosphere is at its highest |
| Declining (0.65) | Many strong storms happen in this phase |

| Season | Why |
|---|---|
| Winter (15 January) | At mid-latitudes, storms often **raise** the ionosphere in winter (positive storms) |
| Spring equinox (21 March) | The day used in Section 10 |
| Summer (21 June) | Storms often **lower** the ionosphere in summer (negative storms), because the storm-heated air changes its make-up |
| Autumn equinox (23 September) | Equinoxes have the most storms |

**Why a different model is needed here.** In the model of Section 10, the storm effect was not allowed to depend on the
season or the solar phase, so it would give the same answer on all 16 days. Here two more terms are added: one lets the
storm effect change with the season (te(Dst, season)), the other with the solar phase (te(Dst, phase)). Each of these is
also allowed to change with CO₂, so the CO₂ modulation can be different on each kind of day.
The CNN gives the same table in Section 12, as an independent check.

In [ ]:
from pygam import te
PHASES = {"minimum (0.05)": 0.05, "rising (0.20)": 0.20, "maximum (0.40)": 0.40, "declining (0.65)": 0.65}
SEASONS = {"winter (15 Jan)": 15, "spring equinox (21 Mar)": 80, "summer (21 Jun)": 172, "autumn equinox (23 Sep)": 266}
SCEN = [(pn, sn) for pn in PHASES for sn in SEASONS]

def ext_terms(K):
    # The Section 10 model, plus a storm effect that changes with season and with solar phase, and with CO2 in each
    t, _ = build_terms("by", K)
    season = dict(basis=["ps", "cp"], edge_knots=[None, [0.5, 366.5]])
    return (t + te(1, 3, n_splines=[5, 6], **season) + te(1, 3, n_splines=[4, 5], by=4, **season)
              + te(1, 0, n_splines=[5, 6]) + te(1, 0, n_splines=[4, 5], by=4))

def fit_ext(X, y, settings):
    gam = LinearGAM(ext_terms(settings[0])); gam.lam = settings[1]; gam.fit(X, y)
    return gam

def scenario_values(predict):
    # For each kind of day: storm response at 350 ppm, at 430 ppm, and the change per +10 ppm
    out = []
    for pn, sn in SCEN:
        ph, dy = PHASES[pn], SEASONS[sn]
        resp = {c: predict(ref_rows(MAIN_LEVEL, c, phase=ph, doy=dy))[0] - predict(ref_rows(DST_QUIET, c, phase=ph, doy=dy))[0]
                for c in (350.0, 430.0)}
        out.append([resp[350.0], resp[430.0], (resp[430.0] - resp[350.0]) / 8.0])    # average change per 10 ppm over 350 to 430
    return np.array(out)

def scenario_table(point, boots, p):
    lo, hi = np.percentile(boots, [2.5, 97.5], axis=0) if boots is not None else (None, None)
    rows = []
    for i, (pn, sn) in enumerate(SCEN):
        r = {"solar phase": pn, "season": sn,
             "storm response, 350 ppm": point[i, 0], "storm response, 430 ppm": point[i, 1],
             "change per +10 ppm": point[i, 2]}
        if lo is not None:
            r["95% interval"] = f"[{lo[i, 2]:+.3f}, {hi[i, 2]:+.3f}]"
            r["interval excludes zero"] = bool(lo[i, 2] > 0 or hi[i, 2] < 0)
        rows.append(r)
    t = pd.DataFrame(rows).set_index(["solar phase", "season"]); t.columns.name = f"{p} ({PARAMS[p]})"
    return t

scen_tables, ext_models = {}, {}
t0 = time.time()
for p in PARAMS:
    st = data_store[p]; X, y, years = st["X"], st["y"], st["years"]
    gam = fit_ext(X, y, SET[p]); ext_models[p] = gam
    point = scenario_values(gam.predict)
    rng = np.random.default_rng(SEED); uniq = np.unique(years); where = {u: np.where(years == u)[0] for u in uniq}
    boots = []
    for _ in range(N_BOOT_CHECK):
        pick = np.concatenate([where[u] for u in rng.choice(uniq, len(uniq))])
        boots.append(scenario_values(fit_ext(X[pick], y[pick], SET[p]).predict))
    scen_tables[p] = savetab(scenario_table(point, np.array(boots), p).round(3), f"table_scenarios_gam_{p}")
    print(f"{p}: deviance explained {gam_stats(gam)['deviance explained']:.3f}")
    display(scen_tables[p])
print(f"Time: {time.time() - t0:.0f} s")

In [ ]:
# Picture of the table: storm response at 350 and 430 ppm on each kind of day
for p in PARAMS:
    t = scen_tables[p]
    fig, axes = plt.subplots(1, len(PHASES), figsize=(16, 3.8), sharey=True)
    for ax, pn in zip(axes, PHASES):
        sub = t.loc[pn]; xpos = np.arange(len(sub))
        ax.bar(xpos - 0.2, sub["storm response, 350 ppm"], width=0.38, color=COL[0], label="350 ppm")
        ax.bar(xpos + 0.2, sub["storm response, 430 ppm"], width=0.38, color=COL[1], label="430 ppm")
        ax.axhline(0, color="0.3", lw=0.8)
        ax.set_xticks(xpos); ax.set_xticklabels([x.split(" (")[0] for x in sub.index], rotation=20)
        ax.set_title(f"Solar {pn}")
    axes[0].set_ylabel(f"Storm response of {p} ({PARAMS[p]})"); axes[0].legend()
    fig.suptitle(f"{p}: storm response (Dst {MAIN_LEVEL:.0f} nT) at 350 and 430 ppm, by solar phase and season")
    plt.tight_layout(); savefig(f"fig_scenarios_{p}"); plt.show()

In [ ]:
# SHAP on the model with seasonal and solar-phase storm effects
for p in PARAMS:
    st = data_store[p]
    res = run_shap(gam_predictor(ext_models[p]), st["X"][:, :4], NAMES, f"{p}, seasonal storm model", f"{p}_ext", PARAMS[p])
    shap_results[(p, "ext")] = res
    show_shap(res, f"--- {p}, model with seasonal and solar-phase storm effects ---")

## 10c. Does CO₂ change how the ionosphere responds to solar activity?

Space weather has two parts: geomagnetic storms and solar activity. Sections 9 and 10 asked whether CO₂ changes the
response to storms. Here the same question is asked for the Sun, in the same three ways as for Dst:

1. **The interaction surface and "who modifies whom"**: a te(solar cycle phase, CO₂) term is added to the main model.
2. **A number with a 95% interval**: the **solar response** is the predicted value at solar maximum minus the value at
   solar minimum, on a quiet day, at the same CO₂ level. The **solar modulation** is how much that solar response changes
   for every +10 ppm of CO₂. It is calculated in two ways:
   - with **solar cycle phase** (maximum 0.40 against minimum 0.05), as in the original models;
   - with **F10.7** (81-day mean of 180 sfu against 75 sfu), which measures how strong the Sun actually is. This matters because
     the cycles were not equally strong (Section 2), and with phase alone a weaker cycle at a higher CO₂ level could look
     like a CO₂ effect. Every CO₂ range in the record contains both low and high F10.7, so this comparison is inside the data.
3. **SHAP** on both models, with the solar SHAP slope on high-solar days.

In [ ]:
# 1. Interaction surface and who modifies whom, for solar cycle phase and CO2
def te_solar_terms(K):
    t, _ = build_terms("main", K)
    return t + te(0, 2, n_splines=[6, 4])

te_solar = {}
phase_grid = np.linspace(0, 1, 60)
PP, CC2 = np.meshgrid(phase_grid, co2_grid)
for p in PARAMS:
    st = data_store[p]
    g = LinearGAM(te_solar_terms(SET[p][0])); g.lam = SET[p][1]; g.fit(st["X"], st["y"])
    te_solar[p] = g
    XX = np.column_stack([PP.ravel(), np.full(PP.size, DST_QUIET), CC2.ravel(), np.full(PP.size, float(DOY_REF)), (CC2.ravel() - CO2_REF) / 10])
    Z = g.partial_dependence(term=len(g.terms) - 2, X=XX).reshape(PP.shape)    # the te(phase, CO2) term
    vmax = np.abs(Z).max()
    fig = plt.figure(figsize=(17, 4.2))
    ax = fig.add_subplot(1, 3, 1)
    im = ax.pcolormesh(PP, CC2, Z, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto", rasterized=True); ax.grid(False)
    fig.colorbar(im, ax=ax, label=f"Effect on {p} ({PARAMS[p]})")
    ax.set_xlabel("Solar cycle phase"); ax.set_ylabel("CO₂ (ppm)"); ax.set_title("Interaction term")
    ax2 = fig.add_subplot(1, 3, 2)
    for i, c in enumerate(CO2_LEVELS):
        X1 = np.column_stack([phase_grid, np.full(60, DST_QUIET), np.full(60, float(c)), np.full(60, float(DOY_REF)), np.full(60, (c - CO2_REF) / 10)])
        ax2.plot(phase_grid, g.predict(X1), color=COL[i % 5], ls=LS[i % 5], lw=1.8, label=f"CO₂ = {c} ppm")
    ax2.set_xlabel("Solar cycle phase"); ax2.set_ylabel(f"Predicted {p} ({PARAMS[p]})"); ax2.set_title("Case 1: CO₂ as the modifier"); ax2.legend(fontsize=8)
    ax3 = fig.add_subplot(1, 3, 3)
    for i, ph in enumerate([0.05, 0.20, 0.40, 0.65]):
        X1 = ref_rows(DST_QUIET, co2_line, phase=ph)
        ax3.plot(co2_line, g.predict(X1), color=COL[i], ls=LS[i], lw=1.8, label=f"phase = {ph}")
    ax3.set_xlabel("CO₂ (ppm)"); ax3.set_ylabel(f"Predicted {p} ({PARAMS[p]})"); ax3.set_title("Case 2: solar phase as the modifier"); ax3.legend(fontsize=8)
    fig.suptitle(f"{p}: solar cycle phase × CO₂ (quiet day, day of year {DOY_REF})")
    plt.tight_layout(); savefig(f"fig_solar_interaction_{p}"); plt.show()

In [ ]:
# 2. The solar modulation as a number, with 95% intervals
from pygam import s as s_
def boot_numbers(fit_fn, eval_fn, X, y, years, n_boot=N_BOOT):
    gam = fit_fn(X, y); point = eval_fn(gam)
    rng = np.random.default_rng(SEED); uniq = np.unique(years); where = {u: np.where(years == u)[0] for u in uniq}
    boots = []
    for _ in range(n_boot):
        pick = np.concatenate([where[u] for u in rng.choice(uniq, len(uniq))])
        boots.append(eval_fn(fit_fn(X[pick], y[pick])))
    return gam, point, np.array(boots)

SOLAR_HIGH, SOLAR_LOW = 180.0, 75.0
solar_models, solar_rows = {}, []
for p in PARAMS:
    K, lam = SET[p]
    def fit_phase(X, y):
        t, _ = build_terms("by", K); g = LinearGAM(t + s_(0, by=4, n_splines=8)); g.lam = lam; g.fit(X, y); return g
    def eval_phase(g):
        r = lambda c: g.predict(ref_rows(DST_QUIET, c, phase=0.40))[0] - g.predict(ref_rows(DST_QUIET, c, phase=0.05))[0]
        return np.array([r(350.0), r(430.0), r(400.0) - r(390.0)])
    def fit_f107(X, y):
        t, _ = build_terms("by", K, solar=True, drop="phase"); g = LinearGAM(t + s_(5, by=4, n_splines=8)); g.lam = lam; g.fit(X, y); return g
    def eval_f107(g):
        r = lambda c: g.predict(ref_rows(DST_QUIET, c, solar_value=SOLAR_HIGH))[0] - g.predict(ref_rows(DST_QUIET, c, solar_value=SOLAR_LOW))[0]
        return np.array([r(350.0), r(430.0), r(400.0) - r(390.0)])
    st = data_store[p]
    gA, ptA, bA = boot_numbers(fit_phase, eval_phase, st["X"], st["y"], st["years"])
    d, Xf, yf, yrf = get_xy(df, p, solar="f107_81")
    gB, ptB, bB = boot_numbers(fit_f107, eval_f107, Xf, yf, yrf)
    solar_models[p] = dict(phase=gA, f107=gB, Xf=Xf)
    for name, pt, bt, g in [("solar cycle phase (0.40 vs 0.05)", ptA, bA, gA), ("F10.7 (180 vs 75 sfu)", ptB, bB, gB)]:
        lo, hi = np.percentile(bt, [2.5, 97.5], axis=0)
        solar_rows.append({"parameter": p, "solar measure": name,
                           "deviance explained": gam_stats(g)["deviance explained"],
                           "solar response, 350 ppm": fmt(pt[0], lo[0], hi[0], 2),
                           "solar response, 430 ppm": fmt(pt[1], lo[1], hi[1], 2),
                           "relative change 350 to 430 (%)": 100 * (pt[1] - pt[0]) / pt[0],
                           "solar modulation per +10 ppm": fmt(pt[2], lo[2], hi[2], 3),
                           "interval excludes zero": bool(lo[2] > 0 or hi[2] < 0)})
solar_table = pd.DataFrame(solar_rows).set_index(["parameter", "solar measure"])
display(savetab(solar_table.round(3), "table_solar_modulation"))

In [ ]:
# 3. SHAP on both solar models
NAMES_F107 = ["F10.7 (81-day)", "Dst", "CO2", "Season"]
def f107_predictor(gam):
    # SHAP sees F10.7, Dst, CO2, day of year; the model's own column order is rebuilt here (phase column is unused)
    def predict(A):
        A = np.asarray(A, dtype=float)
        return gam.predict(np.column_stack([np.zeros(len(A)), A[:, 1], A[:, 2], A[:, 3], (A[:, 2] - CO2_REF) / 10.0, A[:, 0]]))
    return predict
for p in PARAMS:
    st = data_store[p]
    res = run_shap(gam_predictor(solar_models[p]["phase"]), st["X"][:, :4], NAMES, f"{p}, solar-phase interaction model", f"{p}_solar_phase", PARAMS[p])
    shap_results[(p, "solar_phase")] = res; show_shap(res, f"--- {p}, solar cycle phase × CO2 model ---")
    Xf = solar_models[p]["Xf"]
    X4 = np.column_stack([Xf[:, 5], Xf[:, 1], Xf[:, 2], Xf[:, 3]])
    res = run_shap(f107_predictor(solar_models[p]["f107"]), X4, NAMES_F107, f"{p}, F10.7 interaction model", f"{p}_solar_f107", PARAMS[p])
    shap_results[(p, "solar_f107")] = res; show_shap(res, f"--- {p}, F10.7 × CO2 model ---")

## 11. Can the interaction be seen in the raw data?

As in the original, the model result is checked against the data themselves.

First the coloured scatter plot from the original. Then a check that needs almost no model:

1. Remove solar cycle phase, season and the long-term trend from each parameter, leaving the **anomaly**.
2. Split the record into four groups of days by CO₂ level (quarters).
3. In each quarter, take the mean anomaly on storm days minus the mean anomaly on quiet days.

That difference is the storm response measured directly from the data. If CO₂ changes the storm
response, it should drift up or down from the lowest-CO₂ quarter to the highest.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, p in zip(axes, PARAMS):
    d = df[[DST_COL, "co2_ppm", p]].dropna()
    sc = ax.scatter(d[DST_COL], d["co2_ppm"], c=d[p], s=3, cmap="viridis", rasterized=True)
    fig.colorbar(sc, ax=ax, label=f"{p} ({PARAMS[p]})")
    ax.set_xlabel(DST_LABEL); ax.set_ylabel("CO₂ (ppm)"); ax.grid(False)
fig.suptitle("Each ionospheric parameter across Dst and CO₂ (raw daily values)")
savefig("fig09_raw_coloured_scatter"); plt.show()

In [ ]:
def raw_storm_response(p, n_boot=300):
    st = data_store[p]
    X, y, years = st["X"], st["y"], st["years"]
    # anomaly: remove phase, season and a linear CO2 trend (no Dst in this model)
    g = fit_gam(X, y, "main", SET[p], drop="dst")
    anomaly = y - g.predict(X)
    dst, co2 = X[:, 1], X[:, 2]
    edges = np.quantile(co2, [0, 0.25, 0.5, 0.75, 1.0])
    quarter = np.clip(np.searchsorted(edges, co2, side="right") - 1, 0, 3)
    rng = np.random.default_rng(SEED)
    uniq = np.unique(years); where = {u: np.where(years == u)[0] for u in uniq}
    def diff(index):
        out = []
        for q in range(4):
            sel = index[quarter[index] == q]
            a, b = anomaly[sel][dst[sel] <= STORM_AT], anomaly[sel][dst[sel] > QUIET_ABOVE]
            out.append(a.mean() - b.mean() if len(a) >= 5 and len(b) >= 5 else np.nan)
        return np.array(out)
    point = diff(np.arange(len(y)))
    boots = np.array([diff(np.concatenate([where[u] for u in rng.choice(uniq, len(uniq))])) for _ in range(n_boot)])
    lo, hi = np.nanpercentile(boots, 2.5, axis=0), np.nanpercentile(boots, 97.5, axis=0)
    counts = [int(((quarter == q) & (dst <= STORM_AT)).sum()) for q in range(4)]
    labels = [f"{edges[q]:.0f} to {edges[q + 1]:.0f}" for q in range(4)]
    return pd.DataFrame({"CO2 range (ppm)": labels, "storm days": counts, "storm response": point, "low": lo, "high": hi})

raw_tables = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, p in zip(axes, PARAMS):
    t = raw_storm_response(p); raw_tables[p] = savetab(t.round(3), f"table_raw_storm_response_{p}")
    xpos = np.arange(4)
    ax.errorbar(xpos, t["storm response"], yerr=[t["storm response"] - t["low"], t["high"] - t["storm response"]],
                fmt="o", color=COL[0], capsize=4, ms=7)
    ax.axhline(0, color="0.4", lw=0.8)
    ax.set_xticks(xpos); ax.set_xticklabels(t["CO2 range (ppm)"], rotation=15)
    ax.set_xlabel("CO₂ quarter (ppm)"); ax.set_ylabel(f"Storm-day minus quiet-day anomaly, {p} ({PARAMS[p]})")
fig.suptitle(f"Storm response measured directly from the data (storm: Dst ≤ {STORM_AT:.0f} nT; quiet: Dst > {QUIET_ABOVE:.0f} nT)")
plt.tight_layout(); savefig("fig10_raw_storm_response"); plt.show()
for p in PARAMS:
    print(p); display(raw_tables[p])

## 12. An independent model: the CNN

As in the original, the analysis is repeated with a small neural network, to see whether a
completely different kind of model learns the same thing. The network is the one from the original
notebook. The differences are that it is given the season (as a sine and a cosine of the day of year)
and that whole years are held out for validation.

SHAP is applied to it in the same way as to the pyGAM models, and the storm response at low and
high CO₂ is read from its predictions. The CNN has no bootstrap interval, so treat its numbers as a
cross-check on direction and size.

In [ ]:
cnn_results = {}
if RUN_CNN:
    try:
        import tensorflow as tf
        from tensorflow.keras import Sequential, Input
        from tensorflow.keras.layers import Conv1D, Flatten, Dense, Dropout
        from tensorflow.keras.callbacks import EarlyStopping
        from sklearn.preprocessing import StandardScaler
    except ImportError:
        RUN_CNN = False
        print("TensorFlow is not installed, skipping the CNN section")

In [ ]:
CNN_NAMES = ["Solar cycle phase", "Dst", "CO2", "Season (sin)", "Season (cos)"]

def cnn_inputs(A):
    # phase, Dst, CO2, day of year  ->  phase, Dst, CO2, sin, cos
    A = np.asarray(A, dtype=float); ang = 2 * np.pi * A[:, 3] / 365.25
    return np.column_stack([A[:, 0], A[:, 1], A[:, 2], np.sin(ang), np.cos(ang)])

def train_cnn(p):
    st = data_store[p]
    X5, y = cnn_inputs(st["X"][:, :4]), st["y"].reshape(-1, 1)
    tr, te_ = split_years(st["years"])          # whole years held out for validation
    sx, sy = StandardScaler().fit(X5[tr]), StandardScaler().fit(y[tr])
    tf.keras.utils.set_random_seed(SEED)
    model = Sequential([Input(shape=(1, 5)), Conv1D(16, kernel_size=1, activation="relu"), Flatten(),
                        Dense(32, activation="relu"), Dropout(0.2), Dense(1)])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    hist = model.fit(sx.transform(X5[tr]).reshape(-1, 1, 5), sy.transform(y[tr]), epochs=100, batch_size=32, verbose=0,
                     validation_data=(sx.transform(X5[te_]).reshape(-1, 1, 5), sy.transform(y[te_])),
                     callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)])
    def predict5(B):      # takes the five CNN inputs in physical units
        z = model(sx.transform(np.asarray(B, dtype=float)).reshape(-1, 1, 5), training=False).numpy()
        return sy.inverse_transform(z).ravel()
    return predict5, X5, hist, r2_score(y[te_].ravel(), predict5(X5[te_]))

if RUN_CNN:
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
    for ax, p in zip(axes, PARAMS):
        predict5, X5, hist, r2 = train_cnn(p)
        cnn_results[p] = dict(predict=predict5, X5=X5, r2=r2)
        ax.plot(hist.history["loss"], color=COL[0], label="training years")
        ax.plot(hist.history["val_loss"], color=COL[1], ls="--", label="held-out years")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (MSE, standardised)"); ax.set_title(f"{p}: R² = {r2:.2f} on held-out years"); ax.legend()
    savefig("fig11_cnn_training"); plt.show()

In [ ]:
if RUN_CNN:
    rows = []
    for p in PARAMS:
        r = cnn_results[p]
        res = run_shap(r["predict"], r["X5"], CNN_NAMES, f"{p}, CNN", f"{p}_cnn", PARAMS[p])
        imp = res["importance"]                       # add the two season columns together
        season = imp.loc[["Season (sin)", "Season (cos)"]].sum()
        imp = pd.concat([imp.drop(["Season (sin)", "Season (cos)"]), pd.DataFrame([season], index=["Season"])])
        res["importance"] = imp
        shap_results[(p, "cnn")] = res
        show_shap(res, f"--- {p}, CNN ---")
        # storm response read from the CNN's predictions
        for c in (350, 430):
            quiet = r["predict"](cnn_inputs(ref_rows(DST_QUIET, c)[:, :4]))[0]
            storm = r["predict"](cnn_inputs(ref_rows(MAIN_LEVEL, c)[:, :4]))[0]
            rows.append({"parameter": p, "CO2 (ppm)": c, "quiet day": quiet, "storm day": storm, "storm response": storm - quiet})
    cnn_table = pd.DataFrame(rows).set_index(["parameter", "CO2 (ppm)"])
    cnn_table["modulation per +10 ppm"] = np.nan
    for p in PARAMS:
        m = (cnn_table.loc[(p, 430), "storm response"] - cnn_table.loc[(p, 350), "storm response"]) / 8.0
        cnn_table.loc[(p, 430), "modulation per +10 ppm"] = m
    display(savetab(cnn_table.round(3), "table_cnn_storm_response"))

In [ ]:
# The CNN on the same 16 kinds of day as Section 10b, and its solar response
if RUN_CNN:
    cnn_scen, cnn_solar = {}, []
    for p in PARAMS:
        pred = lambda R, f=cnn_results[p]["predict"]: f(cnn_inputs(R[:, :4]))
        cnn_scen[p] = savetab(scenario_table(scenario_values(pred), None, p).round(3), f"table_scenarios_cnn_{p}")
        print(f"--- {p}, CNN ---"); display(cnn_scen[p])
        r = lambda c: pred(ref_rows(DST_QUIET, c, phase=0.40))[0] - pred(ref_rows(DST_QUIET, c, phase=0.05))[0]
        cnn_solar.append({"parameter": p, "solar response, 350 ppm": r(350.0), "solar response, 430 ppm": r(430.0),
                          "solar modulation per +10 ppm": (r(430.0) - r(350.0)) / 8.0})
    display(savetab(pd.DataFrame(cnn_solar).set_index("parameter").round(3), "table_cnn_solar_response"))

## 12b. The Case 1 and Case 2 plots on the new models, and the width of each curve

This repeats the plots from the original analysis, on every new model:

- **Case 1 (CO₂ as the modifier):** Dst against the predicted value, one plot for each CO₂ level. The CO₂ levels are the same
  percentiles as in the original: minimum, 1st to 5th, 25th, 50th, 75th, 80th, 90th, 95th, 97th percentile and maximum.
- **Case 2 (Dst as the modifier):** CO₂ against the predicted value, one plot for each Dst level, at the same Dst percentiles
  as the original (maximum, 99th, 98th, 97th, 95th, 90th, 80th, 75th, 50th, 25th, 5th to 1st, minimum).

As in the original, solar cycle phase is fixed at 0. The season is fixed at the March equinox.

**The width of each curve.** For each curve, the width is its highest value minus its lowest value. In Case 1 this is how
much Dst can move the ionosphere, **up or down**, at that CO₂ level. It is the direction-free measure of the storm effect:
a curve that swings up by 3 and one that swings down by 3 both have a width of 3. If CO₂ dampens the ionosphere's response
to storms, the Case 1 width should get smaller as CO₂ rises.

The Dst range for the width is the 0.5th to 99th percentile of the data, so it does not depend on a handful of extreme days.
For the pyGAM models the width has a 95% range from 100 refits on resampled years. The CNN has one value.

Models: the interaction model (te, Section 9), the linear-in-CO₂ model (by, Section 10), the model with seasonal and
solar-phase storm effects (ext, Section 10b, also shown at solar maximum, phase 0.40), and the CNN (Section 12, also at phase 0.40).

In [ ]:
CO2_PCTS = [0, 1, 2, 3, 4, 5, 25, 50, 75, 80, 90, 95, 97, 100]
DST_PCTS = [100, 99, 98, 97, 95, 90, 80, 75, 50, 25, 5, 4, 3, 2, 1]
# The minimum (one day at about -589 nT, the March 1989 superstorm) is left out: it is far below every other day,
# so the models would be guessing there and give impossible values such as negative TEC.
co2_levels = np.percentile(df["co2_ppm"].dropna(), CO2_PCTS)
dst_levels = np.percentile(df[DST_COL].dropna(), DST_PCTS)
dst_w = np.linspace(np.nanpercentile(df[DST_COL], 0.5), np.nanpercentile(df[DST_COL], 99), 60)    # Dst range for Case 1
co2_w = np.linspace(df["co2_ppm"].min(), df["co2_ppm"].max(), 60)                                   # CO2 range for Case 2
def pct_name(q):
    if q == 0: return "minimum"
    if q == 100: return "maximum"
    suffix = {1: "st", 2: "nd", 3: "rd"}.get(q % 10 if q % 100 not in (11, 12, 13) else 0, "th")
    return f"{q}{suffix} pct"

def case1_curves(predict, phase=0.0):
    return np.array([predict(ref_rows(dst_w, c, phase=phase)) for c in co2_levels])
def case2_curves(predict, phase=0.0):
    return np.array([predict(ref_rows(d, co2_w, phase=phase)) for d in dst_levels])

def plot_cases(c1, c2, p, model_name, tag):
    # Case 1: one panel per CO2 level. Case 2: one panel per Dst level.
    for curves, xs, levels, pcts, xlab, what, case in [
            (c1, dst_w, co2_levels, CO2_PCTS, DST_LABEL, "CO₂", 1),
            (c2, co2_w, dst_levels, DST_PCTS, "CO₂ (ppm)", "Dst", 2)]:
        ncol_ = 4; nrow_ = int(np.ceil(len(levels) / ncol_))
        fig, axes = plt.subplots(nrow_, ncol_, figsize=(16, 2.7 * nrow_), sharex=True, sharey=True)
        for ax, cur, lev, q in zip(axes.ravel(), curves, levels, pcts):
            ax.plot(xs, cur, color=COL[0], lw=1.8)
            unit = "ppm" if what == "CO₂" else "nT"
            ax.set_title(f"{what} {pct_name(q)} ({lev:.1f} {unit}): width {np.ptp(cur):.2f}", fontsize=9)
        for ax in axes.ravel()[len(levels):]:
            ax.axis("off")
        for ax in axes[-1]:
            ax.set_xlabel(xlab)
        for ax in axes[:, 0]:
            ax.set_ylabel(f"Predicted {p}")
        title = "Case 1: Dst against predicted value, CO₂ fixed" if case == 1 else "Case 2: CO₂ against predicted value, Dst fixed"
        fig.suptitle(f"{p}, {model_name}. {title} (phase 0, March)", y=1.0)
        plt.tight_layout(); savefig(f"fig_case{case}_{tag}_{p}"); plt.show()

def refit(kind, X, y, settings):
    return fit_ext(X, y, settings) if kind == "ext" else fit_gam(X, y, kind, settings)

GAM_KINDS = {"te": "interaction model (te)", "by": "linear-in-CO₂ model (by)", "ext": "seasonal storm model (ext)"}
width_rows, width_store = [], {}
t0 = time.time()
for p in PARAMS:
    st = data_store[p]; X, y, years = st["X"], st["y"], st["years"]
    fitted = {"te": te_models[p], "by": by_models[p], "ext": ext_models[p]}
    for kind, label in GAM_KINDS.items():
        g = fitted[kind]
        c1, c2 = case1_curves(g.predict), case2_curves(g.predict)
        plot_cases(c1, c2, p, label, kind)
        phases = (0.0, 0.40) if kind == "ext" else (0.0,)
        rng = np.random.default_rng(SEED); uniq = np.unique(years); where = {u: np.where(years == u)[0] for u in uniq}
        boots = {ph: [] for ph in phases}
        for _ in range(N_BOOT_CHECK):
            pick = np.concatenate([where[u] for u in rng.choice(uniq, len(uniq))])
            gb = refit(kind, X[pick], y[pick], SET[p])
            for ph in phases:
                boots[ph].append(np.ptp(case1_curves(gb.predict, ph), axis=1))
        for ph in phases:
            w = np.ptp(case1_curves(g.predict, ph), axis=1); lo, hi = np.percentile(np.array(boots[ph]), [2.5, 97.5], axis=0)
            width_store[(p, kind, ph)] = (w, lo, hi)
            for q, lev, wi, l_, h_ in zip(CO2_PCTS, co2_levels, w, lo, hi):
                width_rows.append({"parameter": p, "model": kind, "phase": ph, "CO2 level": pct_name(q), "CO2 (ppm)": lev,
                                   "width": wi, "low": l_, "high": h_})
print(f"Time: {time.time() - t0:.0f} s")

In [ ]:
# The CNN, the same way (no refits, so no range)
if RUN_CNN:
    for p in PARAMS:
        pred = lambda R, f=cnn_results[p]["predict"]: f(cnn_inputs(R[:, :4]))
        c1, c2 = case1_curves(pred), case2_curves(pred)
        plot_cases(c1, c2, p, "CNN", "cnn")
        for ph in (0.0, 0.40):
            w = np.ptp(case1_curves(pred, ph), axis=1)
            width_store[(p, "cnn", ph)] = (w, None, None)
            for q, lev, wi in zip(CO2_PCTS, co2_levels, w):
                width_rows.append({"parameter": p, "model": "cnn", "phase": ph, "CO2 level": pct_name(q), "CO2 (ppm)": lev,
                                   "width": wi, "low": np.nan, "high": np.nan})
width_table = pd.DataFrame(width_rows)
savetab(width_table.round(4), "table_case1_widths")

# Summary: width of the Case 1 curve at each CO2 level, for every model
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, p in zip(axes, PARAMS):
    i = 0
    for (pp, kind, ph), (w, lo, hi) in width_store.items():
        if pp != p:
            continue
        lab = f"{kind}, phase {ph}"
        col6 = (COL + ["#7a7a7a"])[i % 6]; mk = "os^Dvx"[i % 6]
        ax.plot(co2_levels, w, color=col6, ls=LS[i % 5], marker=mk, ms=4, lw=1.6, label=lab)
        if lo is not None:
            ax.fill_between(co2_levels, lo, hi, color=col6, alpha=0.10)
        i += 1
    ax.set_xlabel("CO₂ level (ppm)"); ax.set_ylabel(f"Width of the Dst curve, {p} ({PARAMS[p]})"); ax.set_title(p)
axes[0].legend(fontsize=7)
fig.suptitle("Case 1: how much Dst can move the ionosphere (up or down) at each CO₂ level")
plt.tight_layout(); savefig("fig_case1_width_summary"); plt.show()

# A compact table: width at the lowest CO2 level, the median, and the highest, with the change
rows = []
for (p, kind, ph), (w, lo, hi) in width_store.items():
    r = {"parameter": p, "model": kind, "phase": ph,
         "width at minimum CO2": w[0], "width at median CO2": w[CO2_PCTS.index(50)], "width at maximum CO2": w[-1],
         "change minimum to maximum (%)": 100 * (w[-1] - w[0]) / w[0]}
    if lo is not None:
        r["range at minimum CO2"] = f"[{lo[0]:.2f}, {hi[0]:.2f}]"; r["range at maximum CO2"] = f"[{lo[-1]:.2f}, {hi[-1]:.2f}]"
    rows.append(r)
width_summary = pd.DataFrame(rows).set_index(["parameter", "model", "phase"])
display(savetab(width_summary.round(3), "table_case1_width_summary"))

### All curves on one graph

As in the original slides: every Case 1 curve (one per CO₂ level) on one graph, and every Case 2 curve (one per Dst level) on
another, with a legend for each level. One figure per model, with TEC, foF2 and hmF2 side by side.

In [ ]:
# All Case 1 curves on one graph, and all Case 2 curves on one graph, for every parameter and model
from matplotlib import colormaps
OVERLAY_MODELS = {"te": "interaction model (te)", "by": "linear-in-CO₂ model (by)",
                  "ext": "seasonal storm model (ext)", "cnn": "CNN"}
def model_predict(kind, p):
    if kind == "cnn":
        return lambda R, f=cnn_results[p]["predict"]: f(cnn_inputs(R[:, :4]))
    return {"te": te_models, "by": by_models, "ext": ext_models}[kind][p].predict

for kind, label in OVERLAY_MODELS.items():
    if kind == "cnn" and not RUN_CNN:
        continue
    fig, axes = plt.subplots(2, 3, figsize=(19, 10.5))
    for j, p in enumerate(PARAMS):
        pred = model_predict(kind, p)
        c1, c2 = case1_curves(pred), case2_curves(pred)
        cmap1 = colormaps["viridis"](np.linspace(0, 0.95, len(co2_levels)))
        for k, (cur, lev, q) in enumerate(zip(c1, co2_levels, CO2_PCTS)):
            axes[0, j].plot(dst_w, cur, color=cmap1[k], ls=LS[k % 5], lw=1.6,
                            label=f"CO₂ {pct_name(q)} ({lev:.1f} ppm)")
        axes[0, j].set_xlabel(DST_LABEL); axes[0, j].set_ylabel(f"Predicted {p} ({PARAMS[p]})")
        axes[0, j].set_title(f"{p}: Case 1, CO₂ as the modifier")
        cmap2 = colormaps["plasma"](np.linspace(0, 0.9, len(dst_levels)))
        for k, (cur, lev, q) in enumerate(zip(c2, dst_levels, DST_PCTS)):
            axes[1, j].plot(co2_w, cur, color=cmap2[k], ls=LS[k % 5], lw=1.6,
                            label=f"Dst {pct_name(q)} ({lev:.0f} nT)")
        axes[1, j].set_xlabel("CO₂ (ppm)"); axes[1, j].set_ylabel(f"Predicted {p} ({PARAMS[p]})")
        axes[1, j].set_title(f"{p}: Case 2, Dst as the modifier")
    axes[0, 2].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.02, 0.5), title="CO₂ level")
    axes[1, 2].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.02, 0.5), title="Dst level")
    fig.suptitle(f"{label}: all curves on one graph (solar cycle phase 0, 21 March)", fontsize=13)
    plt.tight_layout(); savefig(f"fig_case1_case2_overlay_{kind}"); plt.show()

## 12c. The size of the change, measured from the real data (direction ignored)

The model curves are averages, so a storm that pushes the ionosphere up and one that pushes it down can still cancel
inside them. This section measures the size of the change **directly from the observations**, so nothing can cancel:

1. **Normal level.** For each parameter, a model without Dst (solar cycle phase, CO₂ and season only) is fitted on
   **quiet days only** (Dst above the quiet limit). It is fitted to the logarithm of the parameter, so its expected value is
   never below zero. It gives the value each day would have had without a storm.
2. **Change.** For every day: observed value minus normal level. Then the sign is dropped, giving the **size of the change**,
   in the parameter's own units and as a percentage of the normal level.
3. **Compare by CO₂.** The days are split into four equal groups by CO₂ level. In each group, the mean size of the change is
   worked out for quiet days, storm days (Dst ≤ −50 nT) and strong storm days (Dst ≤ −100 nT), with 95% ranges from resampling whole years.
   Quiet days show the normal day-to-day variation; the storm-day values show how much bigger the change is during storms.
4. **One number.** The model of Section 10 is fitted to the size of the change instead of the value itself. The storm part
   (size on a −100 nT day minus size on a quiet day) at each CO₂ level, and how it changes per +10 ppm, give the answer with a 95% range.
5. **Up or down?** The share of storm days on which the parameter went **up**, in each CO₂ group.

If CO₂ dampens the ionosphere's response to storms, the storm-day size of change should get smaller from the low-CO₂ group
to the high-CO₂ group, while the share of up and down storms may stay the same.

In [ ]:
mag_store, mag_rows, share_rows, mag_number_rows = {}, [], [], []
for p in PARAMS:
    st = data_store[p]; X, y, years = st["X"], st["y"], st["years"]
    quiet = X[:, 1] > QUIET_ABOVE
    gb = fit_gam(X[quiet], np.log(y[quiet]), "main", SET[p], drop="dst")      # normal level, from quiet days
    normal = np.exp(gb.predict(X))
    change = y - normal
    size, size_pct = np.abs(change), 100 * np.abs(change) / normal
    mag_store[p] = dict(normal=normal, change=change, size=size, size_pct=size_pct)
    dst, co2 = X[:, 1], X[:, 2]
    edges = np.quantile(co2, [0, 0.25, 0.5, 0.75, 1]); grp = np.clip(np.searchsorted(edges, co2, side="right") - 1, 0, 3)
    labels = [f"{edges[k]:.0f}-{edges[k + 1]:.0f} ppm" for k in range(4)]
    rng = np.random.default_rng(SEED); uniq = np.unique(years); where = {u: np.where(years == u)[0] for u in uniq}
    picks = [np.concatenate([where[u] for u in rng.choice(uniq, len(uniq))]) for _ in range(300)]
    for cond, sel in [("quiet", dst > QUIET_ABOVE), ("storm (Dst <= -50)", dst <= -50), ("strong storm (Dst <= -100)", dst <= -100)]:
        for k in range(4):
            m = sel & (grp == k)
            b = [size[pk][m[pk]].mean() for pk in picks if m[pk].sum() > 3]
            bp = [size_pct[pk][m[pk]].mean() for pk in picks if m[pk].sum() > 3]
            mag_rows.append({"parameter": p, "days": cond, "CO2 group": labels[k], "n": int(m.sum()),
                             "mean size of change": size[m].mean(), "low": np.percentile(b, 2.5), "high": np.percentile(b, 97.5),
                             "mean size of change (%)": size_pct[m].mean(), "low (%)": np.percentile(bp, 2.5), "high (%)": np.percentile(bp, 97.5)})
    stormy = dst <= -50
    for k in range(4):
        m = stormy & (grp == k)
        share_rows.append({"parameter": p, "CO2 group": labels[k], "storm days": int(m.sum()),
                           "share that went up (%)": 100 * (change[m] > 0).mean(), "share that went down (%)": 100 * (change[m] < 0).mean()})
    for lab, target in [("size of change", size), ("size of change (%)", size_pct)]:
        gam, point, boots = bootstrap_by_model(X, target, years, SET[p])
        r, rb = response(point, MAIN_LEVEL), response(boots, MAIN_LEVEL)
        j350, j430 = int(np.where(CO2_BOOT == 350)[0][0]), int(np.where(CO2_BOOT == 430)[0][0])
        lo, hi = ci(modulation(boots, MAIN_LEVEL))
        mag_number_rows.append({"parameter": p, "measure": lab,
                                "extra size on a storm day, 350 ppm": fmt(r[j350], *ci(rb[:, j350]), 2),
                                "extra size on a storm day, 430 ppm": fmt(r[j430], *ci(rb[:, j430]), 2),
                                "change per +10 ppm": fmt(modulation(point, MAIN_LEVEL), lo, hi, 3),
                                "interval excludes zero": bool(lo > 0 or hi < 0)})
mag_table = savetab(pd.DataFrame(mag_rows).set_index(["parameter", "days", "CO2 group"]).round(3), "table_size_of_change_by_co2")
share_table = savetab(pd.DataFrame(share_rows).set_index(["parameter", "CO2 group"]).round(1), "table_storm_up_down_share")
mag_numbers = savetab(pd.DataFrame(mag_number_rows).set_index(["parameter", "measure"]), "table_size_of_change_numbers")
display(mag_table); display(share_table); display(mag_numbers)

In [ ]:
# Figures: size of the change by CO2 group (quiet, storm, strong storm), and the share of up and down storms
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for j, p in enumerate(PARAMS):
    for row, (col, lo_c, hi_c, unit) in enumerate([("mean size of change", "low", "high", PARAMS[p]),
                                                   ("mean size of change (%)", "low (%)", "high (%)", "%")]):
        ax = axes[row, j]
        for i, cond in enumerate(["quiet", "storm (Dst <= -50)", "strong storm (Dst <= -100)"]):
            t = mag_table.loc[(p, cond)]
            xpos = np.arange(4) + (i - 1) * 0.22
            ax.errorbar(xpos, t[col], yerr=[t[col] - t[lo_c], t[hi_c] - t[col]], fmt="o", ms=6, capsize=3, color=COL[i], label=cond)
        ax.set_xticks(range(4)); ax.set_xticklabels(t.index, rotation=15, fontsize=8)
        ax.set_ylabel(f"Mean size of change ({unit})"); ax.set_title(p)
axes[0, 0].legend(fontsize=8)
fig.suptitle("Size of the change from the normal level (up or down), by CO₂ group, from the observations")
plt.tight_layout(); savefig("fig_size_of_change_by_co2"); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, p in zip(axes, PARAMS):
    t = share_table.loc[p]
    ax.bar(range(4), t["share that went up (%)"], color=COL[1], label="went up")
    ax.bar(range(4), t["share that went down (%)"], bottom=t["share that went up (%)"], color=COL[0], label="went down")
    ax.set_xticks(range(4)); ax.set_xticklabels(t.index, rotation=15, fontsize=8); ax.set_title(p)
axes[0].set_ylabel("Share of storm days (%)"); axes[0].legend()
fig.suptitle("On storm days (Dst ≤ −50 nT), did the parameter go up or down from its normal level?")
plt.tight_layout(); savefig("fig_storm_up_down_share"); plt.show()

## 12d. The same for solar activity, measured from the real data

Section 12c asked whether CO₂ changes how much **storms** move the ionosphere. This asks the same about the **Sun**,
straight from the observations:

1. Only quiet days are used (Dst above the quiet limit), so storms are not mixed in.
2. The days are split into four equal groups by CO₂ level.
3. In each group, a straight line is fitted between the parameter and F10.7 (81-day mean), with the season removed by
   sine and cosine terms. The slope is the **solar sensitivity**: how much the parameter rises for every 100 sfu of F10.7.
   The Sun's effect always pushes the same way (more solar radiation, more electrons, a higher layer), so this is already a
   size, with no up-and-down cancelling.
4. One number for all days: the same fit with an F10.7 × CO₂ term gives how much the sensitivity changes per +10 ppm.
   95% ranges come from resampling whole years.

If CO₂ weakens the Sun's effect on the ionosphere, the sensitivity should fall from the low-CO₂ group to the high-CO₂ group.

**Two versions.** The groups do not all contain the same range of solar activity: the middle groups fall mostly in the weak
cycle 24, where F10.7 rarely went above about 150. The ionosphere does not respond to F10.7 in a perfectly straight line,
so a group with stronger solar maxima can give a different slope for that reason alone. The test is therefore done twice:
with all quiet days, and with only F10.7 ≤ 145 sfu, the range every group shares.

In [ ]:
ang = 2 * np.pi * df.index.dayofyear / 365.25
season_cols = np.column_stack([np.sin(ang), np.cos(ang), np.sin(2 * ang), np.cos(2 * ang)])
df_s = df.assign(s1=season_cols[:, 0], c1=season_cols[:, 1], s2=season_cols[:, 2], c2=season_cols[:, 3])

def sensitivity(d, p):
    A = np.column_stack([np.ones(len(d)), d["f107_81"], d["s1"], d["c1"], d["s2"], d["c2"]])
    return 100 * np.linalg.lstsq(A, d[p].values, rcond=None)[0][1]          # per 100 sfu

def sensitivity_change(d, p):
    c = (d["co2_ppm"] - CO2_REF) / 10
    A = np.column_stack([np.ones(len(d)), d["f107_81"], c, d["f107_81"] * c, d["s1"], d["c1"], d["s2"], d["c2"]])
    return 100 * np.linalg.lstsq(A, d[p].values, rcond=None)[0][3]          # change per +10 ppm, per 100 sfu

def year_boot(d, fn, n=300):
    rng = np.random.default_rng(SEED); yrs = d["year"].unique(); by_year = {y: d[d["year"] == y] for y in yrs}
    return np.array([fn(pd.concat([by_year[y] for y in rng.choice(yrs, len(yrs))])) for _ in range(n)])

sol_rows, sol_num_rows = [], []
for version, fmax in [("all quiet days", np.inf), ("F10.7 <= 145 sfu only", 145.0)]:
    for p in PARAMS:
        d = df_s[(df_s[DST_COL] > QUIET_ABOVE) & (df_s["f107_81"] <= fmax)][[p, "f107_81", "co2_ppm", "year", "s1", "c1", "s2", "c2"]].dropna()
        edges = np.quantile(d["co2_ppm"], [0, 0.25, 0.5, 0.75, 1]); grp = np.clip(np.searchsorted(edges, d["co2_ppm"], side="right") - 1, 0, 3)
        for k in range(4):
            dk = d[grp == k]
            b = year_boot(dk, lambda x: sensitivity(x, p))
            sol_rows.append({"version": version, "parameter": p, "CO2 group": f"{edges[k]:.0f}-{edges[k + 1]:.0f} ppm",
                             "quiet days": len(dk), "F10.7 range": f"{dk['f107_81'].min():.0f}-{dk['f107_81'].max():.0f}",
                             "sensitivity per 100 sfu": sensitivity(dk, p),
                             "low": np.percentile(b, 2.5), "high": np.percentile(b, 97.5)})
        b = year_boot(d, lambda x: sensitivity_change(x, p)); lo, hi = np.percentile(b, [2.5, 97.5])
        sol_num_rows.append({"version": version, "parameter": p,
                             "change in sensitivity per +10 ppm (per 100 sfu)": fmt(sensitivity_change(d, p), lo, hi, 3),
                             "interval excludes zero": bool(lo > 0 or hi < 0)})
solar_data_table = savetab(pd.DataFrame(sol_rows).set_index(["version", "parameter", "CO2 group"]).round(3), "table_solar_sensitivity_by_co2")
solar_data_numbers = savetab(pd.DataFrame(sol_num_rows).set_index(["version", "parameter"]), "table_solar_sensitivity_numbers")
display(solar_data_table); display(solar_data_numbers)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, p in zip(axes, PARAMS):
    for i, version in enumerate(["all quiet days", "F10.7 <= 145 sfu only"]):
        t = solar_data_table.loc[(version, p)]
        xpos = np.arange(4) + (i - 0.5) * 0.25
        ax.errorbar(xpos, t["sensitivity per 100 sfu"], yerr=[t["sensitivity per 100 sfu"] - t["low"], t["high"] - t["sensitivity per 100 sfu"]],
                    fmt="o", ms=6, capsize=3, color=COL[i], label=version)
    ax.set_xticks(range(4)); ax.set_xticklabels([f"group {k + 1}" for k in range(4)])
    ax.set_xlabel("CO₂ group (1 = lowest, 4 = highest)"); ax.set_ylabel(f"{p} rise per 100 sfu of F10.7 ({PARAMS[p]})"); ax.set_title(p)
axes[0].legend(fontsize=8)
fig.suptitle("Solar sensitivity on quiet days, by CO₂ group, from the observations")
plt.tight_layout(); savefig("fig_solar_sensitivity_by_co2"); plt.show()

## 13. Check models

The same linear-in-CO₂ interaction model is refitted in six other ways. If the modulation keeps
its sign and size across them, the result is solid. If it changes, the paper has to say so.

| Check | What changes | What it tests |
|---|---|---|
| Main | nothing | the result from Section 10 |
| + F10.7 (daily) | daily F10.7 added | whether the Sun's actual strength explains the result |
| + F10.7 (81-day mean) | smoothed F10.7 added | the same, with the smoother index usually used for the ionosphere |
| 2004 to 2026 only | early, thinly sampled years removed | whether the result depends on the sparse early years |
| 2004 to 2026 + F10.7 | both of the above together | the balanced period with the Sun's strength accounted for |
| Other Dst value | daily mean in place of daily minimum (or the reverse) | whether the result depends on how Dst is summarised |
| No season term | day of year removed | what the season term changes |

In [ ]:
other_dst = "dst_mean" if DST_COL == "dst_min" else "dst_min"
CHECKS = [("Main", dict()),
          ("+ F10.7 (daily)", dict(solar="f107")),
          ("+ F10.7 (81-day mean)", dict(solar="f107_81")),
          ("2004 to 2026 only", dict(start=2004)),
          ("2004 to 2026 + F10.7 (81-day)", dict(start=2004, solar="f107_81")),
          ("No season term", dict(drop="season"))]

def run_check(p, solar=None, start=None, drop=None, n_boot=N_BOOT_CHECK):
    if solar is None and start is None and drop is None:
        gam, point, boots, n_days = by_models[p], by_point[p], by_boots[p], len(data_store[p]["y"])     # Section 10
    else:
        data = df if start is None else df[df["year"] >= start]
        d, X, y, years = get_xy(data, p, solar=solar)
        gam, point, boots = bootstrap_by_model(X, y, years, SET[p], solar=bool(solar), drop=drop, n_boot=n_boot,
                                               solar_value=F107_REF if solar else None)
        n_days = len(y)
    lo_c, hi_c = ci(co2_effect(boots)); lo_m, hi_m = ci(modulation(boots, MAIN_LEVEL))
    return {"days": n_days, "deviance explained": gam_stats(gam)["deviance explained"],
            "CO2 effect per +10 ppm": fmt(co2_effect(point), lo_c, hi_c),
            "CO2 effect interval excludes zero": bool(lo_c > 0 or hi_c < 0),
            f"modulation per +10 ppm (Dst = {MAIN_LEVEL:.0f} nT)": fmt(modulation(point, MAIN_LEVEL), lo_m, hi_m),
            "modulation interval excludes zero": bool(lo_m > 0 or hi_m < 0),
            "_m": modulation(point, MAIN_LEVEL), "_lo": lo_m, "_hi": hi_m}

t0 = time.time(); rows = []
for p in PARAMS:
    for name, kw in CHECKS:
        rows.append({"parameter": p, "check": name, **run_check(p, **kw)})
check_table = pd.DataFrame(rows).set_index(["parameter", "check"])
display(savetab(check_table.drop(columns=["_m", "_lo", "_hi"]).round(3), "table_check_models"))
print(f"Time: {time.time() - t0:.0f} s")

In [ ]:
# The same check with the other way of summarising Dst (its storm levels are different, so it is shown separately)
def other_dst_check(p, n_boot=N_BOOT_CHECK):
    quiet, level = (0.0, -50.0) if other_dst == "dst_mean" else (-10.0, -100.0)
    d, X, y, years = get_xy(df, p, dst_col=other_dst)
    gam = fit_gam(X, y, "by", SET[p])
    def mod(g):
        r = lambda c: g.predict(ref_rows(level, c))[0] - g.predict(ref_rows(quiet, c))[0]
        return r(400.0) - r(390.0)
    rng = np.random.default_rng(SEED); uniq = np.unique(years); where = {u: np.where(years == u)[0] for u in uniq}
    boots = []
    for _ in range(n_boot):
        pick = np.concatenate([where[u] for u in rng.choice(uniq, len(uniq))])
        try:
            boots.append(mod(fit_gam(X[pick], y[pick], "by", SET[p])))
        except Exception:
            continue
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"parameter": p, "Dst used": other_dst, "storm level (nT)": level,
            "modulation per +10 ppm": fmt(mod(gam), lo, hi), "interval excludes zero": bool(lo > 0 or hi < 0)}

other_table = pd.DataFrame([other_dst_check(p) for p in PARAMS]).set_index("parameter")
display(savetab(other_table, "table_check_other_dst"))

In [ ]:
# One picture of all the checks: the modulation and its 95% interval
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, p in zip(axes, PARAMS):
    t = check_table.loc[p]
    ypos = np.arange(len(t))[::-1]
    ax.errorbar(t["_m"], ypos, xerr=[t["_m"] - t["_lo"], t["_hi"] - t["_m"]], fmt="o", color=COL[0], capsize=4, ms=6)
    ax.axvline(0, color="0.4", lw=0.8)
    ax.set_yticks(ypos); ax.set_yticklabels(t.index if ax is axes[0] else [])
    ax.set_xlabel(f"Modulation per +10 ppm CO₂ ({PARAMS[p]})"); ax.set_title(p)
fig.suptitle(f"Change in the storm response (Dst = {MAIN_LEVEL:.0f} nT) per +10 ppm CO₂, in each check model")
plt.tight_layout(); savefig("fig12_check_models"); plt.show()

## 14. Future projection

The projection planned in the original: predicted values at **higher CO₂ levels**, with solar cycle
phase held fixed and one line for each storm level.

Read these as a **scenario**, not a forecast:

- CO₂ above the highest observed value is outside what the model has seen. The model continues its
  straight-line CO₂ trend there, which is the reason the linear-in-CO₂ model is used for this step.
- Solar cycle phase and season are fixed at the reference values. A different phase shifts all the lines together.
- The shaded band is the 95% interval. It widens beyond the observed range, as it should.

The year in which each CO₂ level is reached is estimated from the recent growth rate in the data.

In [ ]:
annual = df["co2_ppm"].resample("YS").mean().dropna()
growth = (annual.iloc[-2] - annual.iloc[-12]) / 10.0            # ppm per year over the last ten full years
last_year, last_co2 = annual.index[-2].year, annual.iloc[-2]
year_of = lambda c: int(round(last_year + (c - last_co2) / growth))
print(f"Recent CO2 growth: {growth:.2f} ppm per year. Highest observed daily value: {df['co2_ppm'].max():.1f} ppm")

co2_max = df["co2_ppm"].max()
proj_rows = []
for p in PARAMS:
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    for i, level in enumerate([DST_QUIET] + DST_LEVELS):
        k = int(np.argmin(np.abs(DST_BOOT - level)))
        line, (lo, hi) = by_point[p][k], ci(by_boots[p][:, k, :])
        ax.plot(CO2_BOOT, line, color=COL[i], ls=LS[i], lw=2, label=f"Dst = {level:.0f} nT")
        ax.fill_between(CO2_BOOT, lo, hi, color=COL[i], alpha=0.12)
        for c in CO2_FUTURE:
            j = int(np.where(CO2_BOOT == c)[0][0])
            proj_rows.append({"parameter": p, "Dst (nT)": level, "CO2 (ppm)": c, "about year": year_of(c),
                              "projected value": line[j], "low": lo[j], "high": hi[j]})
    ax.axvspan(co2_max, CO2_BOOT.max(), color="0.5", alpha=0.12)
    ax.text(co2_max + 2, ax.get_ylim()[1], "beyond the observed range", va="top", fontsize=9, color="0.3")
    ax.set_xlabel("CO₂ (ppm)"); ax.set_ylabel(f"Predicted {p} ({PARAMS[p]})")
    ax.set_title(f"{p}: projection at solar cycle phase {PHASE_REF}, day of year {DOY_REF}"); ax.legend(loc="lower left")
    savefig(f"fig13_projection_{p}"); plt.show()

proj_table = pd.DataFrame(proj_rows).set_index(["parameter", "Dst (nT)", "CO2 (ppm)"])
display(savetab(proj_table.round(2), "table_projection"))

In [ ]:
# Projected storm response: how big is the storm effect itself at future CO2 levels?
rows = []
for p in PARAMS:
    r, rb = response(by_point[p], MAIN_LEVEL), response(by_boots[p], MAIN_LEVEL)
    for c in [350, 390, 430] + CO2_FUTURE:
        j = int(np.where(CO2_BOOT == c)[0][0]); lo, hi = ci(rb[:, j])
        rows.append({"parameter": p, "CO2 (ppm)": c, "observed range": bool(c <= co2_max),
                     f"storm response at Dst = {MAIN_LEVEL:.0f} nT": fmt(r[j], lo, hi, 2)})
display(savetab(pd.DataFrame(rows).set_index(["parameter", "CO2 (ppm)"]), "table_projected_storm_response"))

## Figures for the paper that combine several results

In [ ]:
from PIL import Image
plt.rcParams.update({"savefig.dpi":300,"font.size":10,"axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,"grid.alpha":0.25,"legend.frameon":False})
P=["TEC","foF2","hmF2"]; V=["phase","season","co2","dst"]; VN={"phase":"Solar cycle phase","season":"Season","co2":"CO₂","dst":"Dst"}
va=pd.read_csv("tables/table_variable_assessment.csv"); sh=pd.read_csv("tables/table_shap_shares.csv"); sh=sh[sh.model=="main"]
fig,axes=plt.subplots(1,3,figsize=(16,4.2))
x=np.arange(3); w=0.2
for i,v in enumerate(V):
    t=va[va.removed==v].set_index("parameter").loc[P]
    axes[0].bar(x+(i-1.5)*w,t["loss in deviance explained (%)"],w,color=COL[i],label=VN[v])
    axes[1].bar(x+(i-1.5)*w,t["loss in R2 on unseen years"],w,color=COL[i],label=VN[v])
    col={"phase":"Solar cycle phase","season":"Season","co2":"CO2","dst":"Dst"}[v]
    axes[2].bar(x+(i-1.5)*w,sh.set_index("parameter").loc[P][col],w,color=COL[i],label=VN[v])
for ax,t,yl in zip(axes,["(a) Loss in deviance explained when removed","(b) Loss in R² on unseen years when removed","(c) Share of mean |SHAP| (main model)"],["Percentage points","R²","%"]):
    ax.set_xticks(x); ax.set_xticklabels(P); ax.set_title(t,loc="left",fontsize=10); ax.set_ylabel(yl)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig("figures/fig_relative_contribution.png",bbox_inches="tight"); plt.savefig("figures/fig_relative_contribution.pdf",bbox_inches="tight")

lag=pd.read_csv("tables/table_lagged_dst.csv").set_index("parameter").loc[P]
so=pd.read_csv("tables/table_storm_days_only.csv")
fig,axes=plt.subplots(1,2,figsize=(14,4))
axes[0].bar(x-0.18,lag["deviance explained, same-day Dst"],0.36,color=COL[0],label="Same-day Dst")
axes[0].bar(x+0.18,lag["deviance explained, lagged Dst"],0.36,color=COL[1],label="Previous day's Dst")
axes[0].set_ylim(0.7,0.85); axes[0].set_xticks(x); axes[0].set_xticklabels(P); axes[0].set_ylabel("Deviance explained"); axes[0].legend(fontsize=8)
axes[0].set_title("(a) Same-day against previous day's Dst",loc="left",fontsize=10)
for i,v in enumerate(V):
    t=so[so.removed==v].set_index("parameter").loc[P]
    axes[1].bar(x+(i-1.5)*w,t["loss in deviance explained (%)"],w,color=COL[i],label=VN[v])
axes[1].set_xticks(x); axes[1].set_xticklabels(P); axes[1].set_ylabel("Percentage points"); axes[1].legend(fontsize=8)
axes[1].set_title("(b) Storm days only: loss in deviance explained when each variable is removed",loc="left",fontsize=10)
plt.tight_layout(); plt.savefig("figures/fig_lag_and_storm_only.png",bbox_inches="tight"); plt.savefig("figures/fig_lag_and_storm_only.pdf",bbox_inches="tight")

ims=[Image.open(f"figures/fig_shap_summary_{p}_main.png").convert("RGB") for p in P]
h=min(i.height for i in ims); rs=[i.resize((int(i.width*h/i.height),h)) for i in ims]
o=Image.new("RGB",(sum(r.width for r in rs),h),"white"); xx=0
for r in rs: o.paste(r,(xx,0)); xx+=r.width
o.save("figures/fig_shap_summary_main_all.png"); o.save("figures/fig_shap_summary_main_all.pdf")
print("ok")

# Partial dependence of all three parameters in one figure
ims = [Image.open(f"figures/fig05_partial_dependence_{p}_main.png").convert("RGB") for p in P]
w = 2400; rs = [i.resize((w, int(i.height * w / i.width))) for i in ims]
o = Image.new("RGB", (w, sum(r.height for r in rs)), "white"); yy = 0
for r in rs: o.paste(r, (0, yy)); yy += r.height
o.save("figures/fig_partial_dependence_all.png"); o.save("figures/fig_partial_dependence_all.pdf")

## 15. All the numbers for the Results section

Everything the paper needs to quote, in one place. The same tables are saved as CSV files in the
`tables` folder and every figure is saved as PNG and PDF in the `figures` folder.

In [ ]:
print("=" * 100)
print(f"SETTINGS: Dst = {DST_COL}; quiet = {DST_QUIET:.0f} nT; headline storm level = {MAIN_LEVEL:.0f} nT; "
      f"phase = {PHASE_REF}; day of year = {DOY_REF}; bootstrap repeats = {N_BOOT}")
print("=" * 100)

print("\n1. MODEL FIT (main model)")
display(fit_table[["days", "deviance explained", "R2 same data", "R2 unseen years", "RMSE unseen years"]].round(3))

print("\n2. RELATIVE CONTRIBUTION OF EACH VARIABLE (share of mean |SHAP|, %)")
kinds = [k for k in ["main", "te", "by", "ext", "solar_phase", "cnn"] if any(key[1] == k for key in shap_results)]
share = pd.concat({(p, k): shap_results[(p, k)]["importance"]["share (%)"] for p in PARAMS for k in kinds}, axis=1).T
share.index.names = ["parameter", "model"]
display(savetab(share.round(1), "table_shap_shares"))

print("\n3. LOSS WHEN EACH VARIABLE IS REMOVED (main model)")
display(assess[["loss in deviance explained (%)", "loss in R2 on unseen years"]].round(3))

print("\n4. THE MODULATION (linear-in-CO2 interaction model, year-block bootstrap)")
for p in PARAMS:
    display(number_tables[p])

print("\n5. THE MODULATION SEEN BY SHAP (storm days: change in the Dst SHAP value per +10 ppm CO2)")
rows = []
for (p, k), res in shap_results.items():
    sl, lo, hi = res["dst_shap_slope_per_10ppm"]
    rows.append({"parameter": p, "model": k, "storm days explained": res["storm_days_explained"],
                 "slope per +10 ppm": fmt(sl, lo, hi), "CO2 -> Dst's impact (MI)": round(res["mi"]["CO2 -> Dst's impact"], 3),
                 "Dst -> CO2's impact (MI)": round(res["mi"]["Dst -> CO2's impact"], 3)})
display(savetab(pd.DataFrame(rows).set_index(["parameter", "model"]), "table_shap_modulation"))

print("\n6. THE CHECK MODELS")
display(check_table.drop(columns=["_m", "_lo", "_hi"]).round(3))
display(other_table)

print("\n7. STORM RESPONSE MEASURED DIRECTLY FROM THE DATA, BY CO2 QUARTER")
for p in PARAMS:
    print(p); display(raw_tables[p])

print("\n8. PROJECTION")
display(proj_table.round(2))

print("\n9. THE STORM CALCULATION ON 16 KINDS OF DAY (pyGAM with seasonal storm effects)")
for p in PARAMS:
    display(scen_tables[p])
if RUN_CNN:
    print("   ... and the CNN on the same days")
    for p in PARAMS:
        display(cnn_scen[p])

print("\n10. DOES CO2 CHANGE THE RESPONSE TO SOLAR ACTIVITY?")
display(solar_table)
if RUN_CNN:
    display(pd.DataFrame(cnn_solar).set_index("parameter").round(3))
rows = []
for (p, k), res in shap_results.items():
    sl, lo, hi = res["solar_shap_slope_per_10ppm"]
    rows.append({"parameter": p, "model": k, "high-solar days explained": res["high_solar_days_explained"],
                 "solar SHAP slope per +10 ppm": fmt(sl, lo, hi), "CO2 -> solar's impact (MI)": round(res["mi"]["CO2 -> solar's impact"], 3)})
display(savetab(pd.DataFrame(rows).set_index(["parameter", "model"]), "table_shap_solar_modulation"))

print("\n11. YOUR CASE 1 PLOTS: WIDTH OF THE DST CURVE AT EACH CO2 LEVEL (direction ignored)")
display(width_summary.round(3))

print("\n12. SIZE OF THE CHANGE FROM THE OBSERVATIONS (direction ignored)")
display(mag_table); display(share_table); display(mag_numbers)

print("\n13. SOLAR SENSITIVITY FROM THE OBSERVATIONS, BY CO2 GROUP")
display(solar_data_table); display(solar_data_numbers)

### How to read the result

- **The CO₂ effect** (first row of each modulation table) is the long-term change in the quiet-day
  value per 10 ppm. Because CO₂ rises steadily with time, this number also contains anything else
  that changed steadily over the years. The paper should describe it as the long-term trend
  expressed per unit of CO₂.
- **The modulation** is the answer to the main question. Quote the estimate with its 95% interval,
  and say whether the interval excludes zero.
- **The check models decide how strongly it can be worded.** A modulation that excludes zero in the
  main model but not when F10.7 is added means the Sun's strength explains part of it, and the paper must say so.
- **The projection** is a scenario at a fixed solar cycle phase and season.